<a href="https://colab.research.google.com/github/Abraham-Na70/B401-Cenim/blob/main/module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-on Module: Human-Robot Interaction Workshop
**B401 Laboratory: Embedded Systems and Robotics**

Welcome to the Human-Robot Interaction (HRI) Workshop. In this integrated hands-on module, you will learn how to bridge human perception (body movements, hand gestures, and natural language instructions) with the physical actuation of modern robotic systems:
1. **Industrial Robot Manipulator (Universal Robots UR5)** for high-precision manipulation tasks (Pick and Place).
2. **Quadruped Mobile Robot (RoboDog)** for dynamic navigation and environmental mobility tasks.

---

### Workshop Workflow:
* **Chapter 0: Environment Setup and Dependency Installation**
  Setting up the Colab runtime environment, installing Computer Vision, Machine Learning, and WebSockets libraries, and downloading the MediaPipe Hand Landmarker model.
* **Chapter 1: Visual Perception and Hand Gesture Recognition (Computer Vision & Machine Learning)**
  Extracting 21 hand joint coordinates via MediaPipe, invariant 42-feature spatial normalization, interactive dataset recording studio, and Random Forest classification.
* **Chapter 2: Complex Robot Commands Using Large Language Models (LLM)**
  Translating natural language instructions into structured action plans using prompt engineering and LLM reasoning.
* **Chapter 3: Hands-on Learning with the UR5 Robotic Manipulator**
  Distributed network architecture, Time-Leased Mutex concurrency management, chessboard workspace coordinates, dwell stabilization filter, and live physical robot teleoperation.
* **Chapter 4: Interaction and Control of Quadruped Robots (RoboDog)**
  Custom hand pose registration (`gestures.json`), nearest-neighbor landmark matching, real-time REST API teleoperation (`/cmd_vel` & `/simple_cmd`), and natural-language LLM control via system prompt & few-shot engineering for the Lite3 RoboDog.

# Chapter 0: Environment Setup and Dependency Installation

## 0.1 Conceptual Context: Self-Contained Computing Environment

Before diving into the Human-Robot Interaction (HRI) workshop, we need to prepare our Google Colab working environment. In this chapter, we install all required software dependencies—spanning Computer Vision libraries (MediaPipe, OpenCV), Machine Learning tools (Scikit-Learn), asynchronous communication protocols (WebSockets), and the Google AI Studio Python SDK (`google-genai`).

Additionally, we download the pre-trained MediaPipe hand landmarker model weights (`hand_landmarker.task`) locally into the Colab environment. This ensures that the hand pose recognition and gesture training in **Chapter 1** can run entirely self-contained (offline) inside your browser without requiring a live connection to the lab's robot server.

In [ ]:
#@title 0.1 Library Dependencies Installation and Hand Detection Model Download { display-mode: "form" }
# 0.1 Library Dependencies Installation and Hand Detection Model Download
# Please do not modify the contents of this cell

import os
import sys

print("[1/2] Checking and installing required libraries...")
!pip install -q websockets mediapipe opencv-python-headless scikit-learn matplotlib joblib openai requests numpy "google-genai==2.12.1" "google-auth==2.49.0"

MODEL_PATH = "hand_landmarker.task"
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"

print("[2/2] Checking MediaPipe Hand Landmarker model availability...")
if not os.path.exists(MODEL_PATH):
    print(f"Downloading model from {MODEL_URL}...")
    !wget -q -O {MODEL_PATH} {MODEL_URL}
    print("Model successfully downloaded.")
else:
    print("Model file is already available in the working directory.")

print("Installation complete. Environment is ready to use.")

# Chapter 1: Visual Perception and Hand Gesture Recognition (Computer Vision & Machine Learning)

## 1.1 Reading Hand Finger Skeletons with MediaPipe

> **Quick Goal:** Convert raw camera video into 21 finger joint coordinates so the computer recognizes gesture shapes without being distracted by clothing colors or background clutter.

* **The Photo vs. Stick Figure Analogy:**
  * When guessing whether a friend is giving a thumbs-up or waving, we only focus on the **positions and bends of their finger bones**.
  * We don't care about the color of their shirt, the wall paint in the room, or their skin tone.
  * A laptop camera captures millions of raw color pixels, which overwhelms computers and slows them down. We simplify this into **21 finger joint coordinates**, just like drawing a stick figure.
* **The Automatic Magnifying Glass Analogy (Normalization):**
  * When your hand moves closer to the camera, it appears huge. When you pull it back, it shrinks.
  * To keep the AI from getting confused, the hand skeleton is automatically scaled to a uniform, standard size.
  * The wrist point is anchored as the origin benchmark `(0, 0)`.

### How It Works Behind the Scenes
* The MediaPipe algorithm detects 21 coordinates `(x, y)` of finger joints from the camera frame.
* Each joint coordinate is offset relative to the wrist position, then scaled by the palm length.
* The result is a consistent 42-number decimal vector, regardless of whether your hand is on the left, right, close, or far from the camera.

> **Processing Pipeline:**  
> `Raw Camera Frame (900,000+ Pixels)` ➔ `21 Finger Joint Coordinates (MediaPipe)` ➔ `Automatic Scale Normalization` ➔ `42 Standard Invariant Numbers (AI-Ready)`

> **Class Discussion Question:** *If you wore thick winter gloves or sat in a dimly lit room, would this stick figure model still be able to detect finger joints? Why or why not?*

In [ ]:
#@title 1.1 Initialize Hand Landmark Detector (Universal MediaPipe) { display-mode: "form" }
# Please do not modify the contents of this cell

import os
import cv2
import numpy as np
import mediapipe as mp

class HandResult:
    """Wrapper to unify output structures across modern and legacy MediaPipe."""
    def __init__(self, landmarks):
        self.multi_hand_landmarks = [landmarks] if landmarks is not None else None

class UniversalHandDetector:
    """
    Flexible hand detector:
    Supports modern MediaPipe (>=0.10.30, Tasks API) and legacy (mp.solutions.hands).
    """
    def __init__(self, model_path: str = "hand_landmarker.task"):
        self.mode = "legacy"
        if hasattr(mp, "solutions") and hasattr(mp.solutions, "hands"):
            self.detector = mp.solutions.hands.Hands(
                static_image_mode=True,
                max_num_hands=1,
                min_detection_confidence=0.6,
                min_tracking_confidence=0.6,
            )
        else:
            self.mode = "tasks"
            if not os.path.exists(model_path):
                import urllib.request
                url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
                print(f"Downloading MediaPipe Tasks model ({model_path})...")
                urllib.request.urlretrieve(url, model_path)

            from mediapipe.tasks import python
            from mediapipe.tasks.python import vision

            base_options = python.BaseOptions(model_asset_path=model_path)
            options = vision.HandLandmarkerOptions(
                base_options=base_options,
                running_mode=vision.RunningMode.IMAGE,
                num_hands=1,
                min_hand_detection_confidence=0.6,
                min_tracking_confidence=0.6,
            )
            self.detector = vision.HandLandmarker.create_from_options(options)

    def detect_landmarks(self, rgb_image: np.ndarray):
        """Accepts RGB image (H, W, 3), returns 21 hand landmark coordinates."""
        if self.mode == "legacy":
            results = self.detector.process(rgb_image)
            if results.multi_hand_landmarks:
                return results.multi_hand_landmarks[0]
            return None
        else:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
            results = self.detector.detect(mp_image)
            if results.hand_landmarks and len(results.hand_landmarks) > 0:
                return results.hand_landmarks[0]
            return None

    def process(self, rgb_image: np.ndarray):
        """Interface method aligned with mp.solutions.hands.process()."""
        lms = self.detect_landmarks(rgb_image)
        return HandResult(lms)


hands_detector = UniversalHandDetector()

GESTURE_LABELS = [
    "POINT",
    "PEACE",
    "THREE_FINGERS",
    "FOUR_FINGERS",
    "OK_SIGN",
    "OPEN_PALM",
    "THUMBS_UP",
    "FIST",
]

def extract_normalized_features(landmarks) -> np.ndarray:
    """
    Converts 21 MediaPipe landmark points into a 42-feature spatial invariant vector:
      1. Translation: shifts wrist (point 0) to (0, 0).
      2. Scale: divides coordinates by distance from wrist to middle finger MCP (point 9).
    """
    if landmarks is None:
        return None
    lms = getattr(landmarks, "landmark", landmarks)
    coords = np.array([[lm.x, lm.y] for lm in lms])
    if len(coords) < 21:
        return None

    wrist = coords[0]
    translated = coords - wrist

    palm_span = np.linalg.norm(translated[9])
    if palm_span < 1e-6:
        palm_span = 1.0
    normalized = translated / palm_span

    return normalized.flatten()

print(f"Universal Hand Detector ({hands_detector.mode.upper()} mode) and 42-feature extraction ready.")


## 1.2 Collecting a Hand Gesture Sample Dataset

> **Quick Goal:** Record custom hand pose samples directly via your laptop webcam to train an AI model to recognize your specific gesture vocabulary.

* **8 Robot Control Commands:**
  * `POINT` (1 index finger) : Navigate to Quadrant 1.
  * `PEACE` (2 fingers: index & middle) : Navigate to Quadrant 2.
  * `THREE_FINGERS` (3 fingers) : Navigate to Quadrant 3.
  * `FOUR_FINGERS` (4 fingers) : Navigate to Quadrant 4.
  * `OK_SIGN` (thumb-index circle) : Grip / Pick up an object (*Pick Sequence*).
  * `OPEN_PALM` (open palm) : Release gripper / Place object into box (*Release Bin*).
  * `THUMBS_UP` (thumbs up) : Return to neutral ready position (*Home / Ready*).
  * `FIST` (clenched fist) : Idle / Hold position (*Hold / Neutral*).
* **Tips for High-Quality Recording:**
  * Click your desired gesture button, watch the 3-second countdown, and hold your hand steady in front of the camera.
  * Slightly rotate or tilt your hand as the camera captures 10 rapid burst frames. This teaches the AI that a slightly tilted thumb is still a thumb!
  * If your laptop webcam has issues, use the **Load Synthetic Baseline** button to populate standard training samples automatically.

### How It Works Behind the Scenes
* The laptop camera streams via the browser (JavaScript WebRTC Canvas) and transfers frames into Google Colab's Python environment.
* Each captured frame immediately extracts 21 joint coordinates and saves them neatly into our dataset table.

> **Dataset Recording Pipeline:**  
> `Select Gesture Button` ➔ `Countdown 3.. 2.. 1..` ➔ `Rapid 10-Frame Burst Capture` ➔ `21 Joint Landmark Extraction` ➔ `Training-Ready Dataset`

> **Class Discussion Question:** *Why is it recommended to gently tilt and shift your hand while recording, rather than holding it completely still like a statue?*

In [ ]:
#@title 1.2 Gesture Dataset Recording Studio (Browser Webcam) { display-mode: "form" }
# Please do not modify the contents of this cell
# 1.2 Gesture Dataset Collection: Interactive 8-Gesture Grid Studio
import time
import base64
import numpy as np
import cv2
from google.colab import output
from IPython.display import JSON, Javascript, display

DATASET_X = []
DATASET_Y = []

GESTURE_CLASSES = [
    ("POINT", "POINT (1 Finger)"),
    ("PEACE", "PEACE (2 Fingers)"),
    ("THREE_FINGERS", "THREE (3 Fingers)"),
    ("FOUR_FINGERS", "FOUR (4 Fingers)"),
    ("OK_SIGN", "OK_SIGN (Pick)"),
    ("OPEN_PALM", "OPEN_PALM (Release)"),
    ("THUMBS_UP", "THUMBS_UP (Home)"),
    ("FIST", "FIST (Neutral/Hold)"),
]

# ── A. Python Callback: Process Burst Frames in Replace/Append Mode ──
def process_burst_payload(gesture_label, frames_b64, mode="replace"):
    """
    Receives frames from JS.
    mode='replace': Clears previous recordings for this gesture before appending new ones.
    mode='append': Appends to existing recordings.
    """
    global DATASET_X, DATASET_Y

    if mode == "replace":
        filtered_x = [x for x, y in zip(DATASET_X, DATASET_Y) if y != gesture_label]
        filtered_y = [y for y in DATASET_Y if y != gesture_label]
        DATASET_X = filtered_x
        DATASET_Y = filtered_y

    valid_count = 0
    annotated_thumbnails = []
    
    # 21 MediaPipe hand joint landmark connections
    HAND_CONNECTIONS = [
        (0, 1), (1, 2), (2, 3), (3, 4),        # Thumb
        (0, 5), (5, 6), (6, 7), (7, 8),        # Index
        (5, 9), (9, 10), (10, 11), (11, 12),   # Middle
        (9, 13), (13, 14), (14, 15), (15, 16), # Ring
        (13, 17), (17, 18), (18, 19), (19, 20),# Pinky
        (0, 17)                                # Palm Base
    ]

    for b64_str in frames_b64:
        try:
            header, encoded = b64_str.split(',', 1) if ',' in b64_str else ('', b64_str)
            img_bytes = base64.b64decode(encoded)
            img_np = np.frombuffer(img_bytes, dtype=np.uint8)
            frame = cv2.imdecode(img_np, flags=cv2.IMREAD_COLOR)
            if frame is None:
                continue
            flipped = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(flipped, cv2.COLOR_BGR2RGB)

            lms = hands_detector.detect_landmarks(rgb)
            if lms is not None:
                feats = extract_normalized_features(lms)
                if feats is not None:
                    DATASET_X.append(feats)
                    DATASET_Y.append(gesture_label)
                    valid_count += 1
                    
                    # Save skeleton visualization preview & 21 joints for the first 3 samples
                    if len(annotated_thumbnails) < 3:
                        h, w, _ = flipped.shape
                        annotated = flipped.copy()
                        for s_idx, e_idx in HAND_CONNECTIONS:
                            if s_idx < len(lms) and e_idx < len(lms):
                                p1 = (int(lms[s_idx][0] * w), int(lms[s_idx][1] * h))
                                p2 = (int(lms[e_idx][0] * w), int(lms[e_idx][1] * h))
                                cv2.line(annotated, p1, p2, (0, 255, 120), 2, cv2.LINE_AA)
                        for pt_idx, pt in enumerate(lms):
                            cx, cy = int(pt[0] * w), int(pt[1] * h)
                            color = (0, 0, 255) if pt_idx == 0 else (0, 255, 255) if pt_idx in (4, 8, 12, 16, 20) else (255, 120, 0)
                            cv2.circle(annotated, (cx, cy), 4, color, -1, cv2.LINE_AA)
                            cv2.circle(annotated, (cx, cy), 5, (255, 255, 255), 1, cv2.LINE_AA)
                        
                        thumb = cv2.resize(annotated, (140, 105))
                        _, buf = cv2.imencode('.jpg', thumb, [cv2.IMWRITE_JPEG_QUALITY, 75])
                        thumb_b64 = "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode('ascii')
                        annotated_thumbnails.append(thumb_b64)
        except Exception:
            continue

    counts = {g[0]: DATASET_Y.count(g[0]) for g in GESTURE_CLASSES}
    total = len(DATASET_Y)
    return JSON({"valid": valid_count, "total": total, "counts": counts, "thumbnails": annotated_thumbnails})

output.register_callback("notebook.ProcessBurstFrames", process_burst_payload)

# ── B. Reset Dataset Callback ──
def py_reset_dataset():
    global DATASET_X, DATASET_Y
    DATASET_X.clear()
    DATASET_Y.clear()
    counts = {g[0]: 0 for g in GESTURE_CLASSES}
    return JSON({"total": 0, "counts": counts})

output.register_callback("notebook.ResetDataset", py_reset_dataset)

# ── C. Fallback: Synthetic Data Generator ──
def py_load_synthetic():
    global DATASET_X, DATASET_Y
    DATASET_X.clear()
    DATASET_Y.clear()

    templates = {
        "FIST": [[0, 0]] + [[0.05 * i, -0.2] for i in range(1, 5)] + [[0.0, -0.35], [0.0, -0.3], [0.0, -0.25], [0.0, -0.2]] * 4,
        "POINT": [[0, 0]] + [[-0.1, -0.1], [-0.15, -0.15], [-0.18, -0.2], [-0.2, -0.25]] + [[0.0, -0.4], [0.0, -0.65], [0.0, -0.85], [0.0, -1.05]] + [[0.1, -0.35], [0.1, -0.3], [0.1, -0.25], [0.1, -0.2]] * 3,
        "PEACE": [[0, 0]] + [[-0.1, -0.1], [-0.15, -0.15], [-0.18, -0.2], [-0.2, -0.25]] + [[-0.08, -0.4], [-0.12, -0.65], [-0.15, -0.85], [-0.18, -1.05]] + [[0.08, -0.4], [0.12, -0.65], [0.15, -0.85], [0.18, -1.05]] + [[0.1, -0.35], [0.1, -0.3], [0.1, -0.25], [0.1, -0.2]] * 2,
        "THREE_FINGERS": [[0, 0]] + [[-0.1, -0.1], [-0.15, -0.15], [-0.18, -0.2], [-0.2, -0.25]] + [[-0.15, -0.4], [-0.2, -0.65], [-0.25, -0.85], [-0.3, -1.05]] + [[0.0, -0.4], [0.0, -0.65], [0.0, -0.85], [0.0, -1.05]] + [[0.15, -0.4], [0.2, -0.65], [0.25, -0.85], [0.3, -1.05]] + [[0.2, -0.35], [0.2, -0.3], [0.2, -0.25], [0.2, -0.2]],
        "FOUR_FINGERS": [[0, 0]] + [[-0.05, -0.1], [-0.08, -0.15], [-0.1, -0.2], [-0.12, -0.22]] + [[-0.2, -0.4], [-0.25, -0.65], [-0.3, -0.85], [-0.35, -1.05]] + [[-0.07, -0.4], [-0.09, -0.65], [-0.1, -0.85], [-0.12, -1.05]] + [[0.07, -0.4], [0.09, -0.65], [0.1, -0.85], [0.12, -1.05]] + [[0.2, -0.4], [0.25, -0.65], [0.3, -0.85], [0.35, -1.05]],
        "OPEN_PALM": [[0, 0]] + [[-0.25, -0.2], [-0.35, -0.3], [-0.45, -0.4], [-0.55, -0.5]] + [[-0.2, -0.4], [-0.25, -0.65], [-0.3, -0.85], [-0.35, -1.05]] + [[-0.07, -0.4], [-0.09, -0.65], [-0.1, -0.85], [-0.12, -1.05]] + [[0.07, -0.4], [0.09, -0.65], [0.1, -0.85], [0.12, -1.05]] + [[0.2, -0.4], [0.25, -0.65], [0.3, -0.85], [0.35, -1.05]],
        "THUMBS_UP": [[0, 0]] + [[-0.1, -0.3], [-0.15, -0.55], [-0.2, -0.8], [-0.25, -1.05]] + [[0.05, -0.25], [0.05, -0.2], [0.05, -0.15], [0.05, -0.1]] * 4,
        "OK_SIGN": [[0, 0]] + [[-0.1, -0.2], [-0.12, -0.3], [-0.1, -0.4], [-0.05, -0.45]] + [[0.0, -0.2], [0.0, -0.3], [0.0, -0.4], [-0.05, -0.45]] + [[0.1, -0.4], [0.12, -0.65], [0.15, -0.85], [0.18, -1.05]] + [[0.2, -0.4], [0.22, -0.65], [0.25, -0.85], [0.28, -1.05]] + [[0.3, -0.35], [0.32, -0.55], [0.35, -0.75], [0.38, -0.95]],
    }

    np.random.seed(42)
    for label, base_coords in templates.items():
        base = np.array(base_coords[:21])
        for _ in range(40):
            noise = np.random.normal(0, 0.025, base.shape)
            jittered = base + noise
            palm_span = np.linalg.norm(jittered[9]) or 1.0
            feats = (jittered / palm_span).flatten()
            DATASET_X.append(feats)
            DATASET_Y.append(label)

    counts = {g[0]: DATASET_Y.count(g[0]) for g in GESTURE_CLASSES}
    return JSON({"total": len(DATASET_Y), "counts": counts})

output.register_callback("notebook.LoadSyntheticBaseline", py_load_synthetic)

# ── D. Reliable Dashboard UI: Auto Hardware Track Release & Resilient Perms ──
DASHBOARD_JS = """
async function initDatasetStudio() {
  // 1. Release ANY orphaned camera stream from prior runs
  if (window.activeDatasetStream) {
    try { window.activeDatasetStream.getTracks().forEach(t => t.stop()); } catch(e){}
    window.activeDatasetStream = null;
  }
  if (window.teleopStream) {
    try { window.teleopStream.getTracks().forEach(t => t.stop()); } catch(e){}
    window.teleopStream = null;
  }

  const existing = document.getElementById('gesture-dashboard-box');
  if (existing) existing.remove();

  const container = document.createElement('div');
  container.id = 'gesture-dashboard-box';
  container.style.fontFamily = '-apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif';
  container.style.background = '#1e1e1e';
  container.style.color = '#fff';
  container.style.padding = '18px';
  container.style.borderRadius = '12px';
  container.style.maxWidth = '680px';
  container.style.boxShadow = '0 4px 14px rgba(0,0,0,0.5)';
  container.style.marginTop = '10px';

  container.innerHTML = `
    <div style="display:flex; justify-content:space-between; align-items:center; border-bottom: 1px solid #333; padding-bottom: 10px; margin-bottom: 12px;">
      <div>
        <h3 style="margin:0; font-size: 18px; color:#00ffcc;">Hand Gesture Dataset Collection Studio</h3>
        <div style="font-size: 12px; color: #aaa; margin-top:3px;">Select gesture -> 3s countdown -> record 10-frame burst.</div>
      </div>
      <div id="dash-total-badge" style="background:#0984e3; padding: 4px 10px; border-radius: 6px; font-weight:bold; font-size:13px;">Total: 0 Samples</div>
    </div>

    <!-- Mode Selector & Controls -->
    <div style="display:flex; justify-content:space-between; align-items:center; background:#2a2a2a; padding:6px 12px; border-radius:6px; margin-bottom:12px; font-size:12px;">
      <div style="display:flex; align-items:center; gap:12px;">
        <span>Recording Mode:</span>
        <label style="cursor:pointer;"><input type="radio" name="recMode" value="replace" checked> Replace</label>
        <label style="cursor:pointer;"><input type="radio" name="recMode" value="append"> Append</label>
      </div>
      <button id="btn-stop-cam" style="background:#c0392b; color:#fff; border:none; padding:4px 10px; border-radius:4px; font-size:11px; cursor:pointer; font-weight:bold;">Turn Off Camera</button>
    </div>

    <div style="display: flex; gap: 16px; margin-bottom: 14px;">
      <div style="position: relative; width: 280px; height: 210px; background: #000; border-radius: 8px; overflow: hidden; flex-shrink: 0; display:flex; align-items:center; justify-content:center;">
        <video id="dash-video" style="width: 100%; height: 100%; object-fit: cover; transform: scaleX(-1); display:none;" autoplay muted playsinline></video>
        <div id="dash-placeholder" style="color: #888; font-size: 12px; text-align: center; padding: 10px;">Camera standby.<br>Click a gesture button to begin.</div>
        <div id="dash-countdown" style="display:none; position: absolute; font-size: 54px; font-weight: bold; color: #f1c40f; text-shadow: 0 2px 8px rgba(0,0,0,0.8);">3</div>
        <div id="dash-rec-bar" style="display:none; position: absolute; bottom: 8px; left: 8px; right: 8px; background: rgba(231,76,60,0.9); color:#fff; font-size:11px; padding: 4px; text-align:center; border-radius: 4px; font-weight:bold;">Recording...</div>
      </div>

      <div style="flex-grow: 1; display: grid; grid-template-columns: 1fr 1fr; gap: 6px; align-content: start;">
        <div class="stat-badge" id="b-POINT" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">POINT: <b>0</b></div>
        <div class="stat-badge" id="b-PEACE" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">PEACE: <b>0</b></div>
        <div class="stat-badge" id="b-THREE_FINGERS" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">THREE: <b>0</b></div>
        <div class="stat-badge" id="b-FOUR_FINGERS" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">FOUR: <b>0</b></div>
        <div class="stat-badge" id="b-OK_SIGN" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">OK_SIGN: <b>0</b></div>
        <div class="stat-badge" id="b-OPEN_PALM" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">PALM: <b>0</b></div>
        <div class="stat-badge" id="b-THUMBS_UP" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">THUMBS: <b>0</b></div>
        <div class="stat-badge" id="b-FIST" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">FIST: <b>0</b></div>
      </div>
    </div>

    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 8px; margin-bottom: 12px;">
      <button class="rec-btn" id="btn-POINT">POINT</button>
      <button class="rec-btn" id="btn-PEACE">PEACE</button>
      <button class="rec-btn" id="btn-THREE_FINGERS">THREE</button>
      <button class="rec-btn" id="btn-FOUR_FINGERS">FOUR</button>
      <button class="rec-btn" id="btn-OK_SIGN">OK_SIGN</button>
      <button class="rec-btn" id="btn-OPEN_PALM">PALM</button>
      <button class="rec-btn" id="btn-THUMBS_UP">THUMBS</button>
      <button class="rec-btn" id="btn-FIST">FIST</button>
    </div>

    <div style="display:flex; justify-content:space-between; align-items:center; border-top:1px solid #333; padding-top:10px;">
      <div style="display:flex; gap:8px;">
        <button id="btn-synth" style="background:#34495e; color:#ecf0f1; border:none; padding:6px 12px; border-radius:5px; font-size:11px; cursor:pointer;">Load Synthetic Baseline</button>
        <button id="btn-reset" style="background:#7f8c8d; color:#ecf0f1; border:none; padding:6px 12px; border-radius:5px; font-size:11px; cursor:pointer;">Reset Data</button>
      </div>
      <div id="dash-info" style="font-size:12px; color:#2ecc71;">Ready to record data.</div>
    </div>
  `;

  document.querySelector('#output-body').appendChild(container);

  const style = document.createElement('style');
  style.textContent = `
    .rec-btn { background: #27ae60; color: white; border: none; padding: 8px 4px; border-radius: 6px; font-size: 11px; font-weight: bold; cursor: pointer; transition: 0.2s; }
    .rec-btn:hover { background: #2ecc71; }
    .rec-btn:disabled { background: #555; cursor: not-allowed; }
  `;
  container.appendChild(style);

  let isRecording = false;

  function stopCamera() {
    if (window.activeDatasetStream) {
      window.activeDatasetStream.getTracks().forEach(t => t.stop());
      window.activeDatasetStream = null;
    }
    const video = document.getElementById('dash-video');
    const placeholder = document.getElementById('dash-placeholder');
    if (video) video.style.display = 'none';
    if (placeholder) {
      placeholder.style.display = 'block';
      placeholder.innerHTML = 'Camera disabled.<br>Click a gesture button to reactivate.';
    }
    const info = document.getElementById('dash-info');
    if (info) { info.innerText = 'Camera turned off.'; info.style.color = '#e74c3c'; }
  }

  function updateBadgeCounts(counts, total) {
    document.getElementById('dash-total-badge').innerText = `Total: ${total} Samples`;
    for (const [key, val] of Object.entries(counts)) {
      const el = document.getElementById(`b-${key}`);
      if (el) {
        const icon = el.innerText.split(' ')[0];
        el.innerHTML = `${icon} ${key.split('_')[0]}: <b>${val}</b>`;
        if (val >= 10) el.style.background = '#1e8449';
        else if (val > 0) el.style.background = '#d35400';
        else el.style.background = '#2d3436';
      }
    }
  }

  async function getCameraStream() {
    if (window.activeDatasetStream && window.activeDatasetStream.active) {
      return window.activeDatasetStream;
    }
    const stream = await navigator.mediaDevices.getUserMedia({
      video: { width: { ideal: 480 }, height: { ideal: 360 }, facingMode: "user" }
    });
    window.activeDatasetStream = stream;
    return stream;
  }

  async function triggerBurst(label) {
    if (isRecording) return;
    isRecording = true;
    container.querySelectorAll('.rec-btn').forEach(b => b.disabled = true);

    const video = document.getElementById('dash-video');
    const placeholder = document.getElementById('dash-placeholder');
    const cd = document.getElementById('dash-countdown');
    const recBar = document.getElementById('dash-rec-bar');
    const info = document.getElementById('dash-info');

    const mode = document.querySelector('input[name="recMode"]:checked')?.value || 'replace';

    placeholder.style.display = 'none';
    video.style.display = 'block';
    info.innerText = `Initializing camera for [${label}]...`;
    info.style.color = '#f1c40f';

    try {
      const stream = await getCameraStream();
      video.srcObject = stream;
      await video.play();

      cd.style.display = 'block';
      for (let c = 3; c > 0; c--) {
        cd.innerText = c;
        await new Promise(r => setTimeout(r, 1000));
      }
      cd.style.display = 'none';

      recBar.style.display = 'block';
      info.innerText = `Recording [${label}]! Slightly tilt/shift your hand...`;
      info.style.color = '#e74c3c';

      const canvas = document.createElement('canvas');
      canvas.width = 400;
      canvas.height = 300;
      const ctx = canvas.getContext('2d');
      const frames = [];

      for (let i = 0; i < 10; i++) {
        ctx.drawImage(video, 0, 0, 400, 300);
        frames.push(canvas.toDataURL('image/jpeg', 0.65));
        recBar.innerText = `Recording Frame ${i+1}/10...`;
        await new Promise(r => setTimeout(r, 200));
      }

      recBar.style.display = 'none';
      info.innerText = `Extracting MediaPipe landmarks for [${label}]...`;
      info.style.color = '#3498db';

      const res = await google.colab.kernel.invokeFunction('notebook.ProcessBurstFrames', [label, frames, mode], {});
      
      let data = null;
      if (res && res.data) {
        if (res.data['application/json']) {
          data = res.data['application/json'];
        } else if (res.data['text/plain']) {
          try {
            data = JSON.parse(res.data['text/plain']);
          } catch(e) {
            try { data = JSON.parse(res.data['text/plain'].replace(/'/g, '"')); } catch(e2){}
          }
        }
      }

      if (data && data.counts) {
        updateBadgeCounts(data.counts, data.total);
        const actionWord = mode === 'replace' ? 'Replace' : 'Append';
        info.innerText = `[${label}]: ${actionWord} successful! (+${data.valid} valid frames).`;
        info.style.color = '#2ecc71';
      } else {
        info.innerText = `Successfully processed (data stored in Python).`;
        info.style.color = '#f1c40f';
      }

      if (data && data.thumbnails && data.thumbnails.length > 0) {
        const box = document.getElementById('dash-joint-box');
        const thumbs = document.getElementById('dash-joint-thumbs');
        if (box && thumbs) {
          box.style.display = 'block';
          thumbs.innerHTML = '';
          data.thumbnails.forEach((src, idx) => {
            const wrap = document.createElement('div');
            wrap.style.textAlign = 'center';
            const img = document.createElement('img');
            img.src = src;
            img.style.borderRadius = '4px';
            img.style.border = '1px solid #00ff78';
            img.style.width = '120px';
            img.style.height = '90px';
            const lbl = document.createElement('div');
            lbl.style.fontSize = '10px';
            lbl.style.color = '#00ff78';
            lbl.style.marginTop = '2px';
            lbl.innerText = `Sample ${idx + 1} (21 Joints)`;
            wrap.appendChild(img);
            wrap.appendChild(lbl);
            thumbs.appendChild(wrap);
          });
        }
      }

    } catch (err) {
      info.innerText = `Camera Error: ${err.message}`;
      info.style.color = '#e74c3c';
    } finally {
      isRecording = false;
      container.querySelectorAll('.rec-btn').forEach(b => b.disabled = false);
    }
  }

  // Bind Buttons
  const gestures = ["POINT", "PEACE", "THREE_FINGERS", "FOUR_FINGERS", "OK_SIGN", "OPEN_PALM", "THUMBS_UP", "FIST"];
  gestures.forEach(g => {
    const btn = document.getElementById(`btn-${g}`);
    if (btn) btn.onclick = () => triggerBurst(g);
  });

  document.getElementById('btn-stop-cam').onclick = stopCamera;

  document.getElementById('btn-synth').onclick = async () => {
    const info = document.getElementById('dash-info');
    info.innerText = "Loading baseline...";
    const res = await google.colab.kernel.invokeFunction('notebook.LoadSyntheticBaseline', [], {});
    let data = res.data['application/json'] || JSON.parse(res.data['text/plain']?.replace(/'/g, '"') || '{}');
    if (data.counts) updateBadgeCounts(data.counts, data.total);
    info.innerText = `Baseline active (${data.total || 320} samples)!`;
    info.style.color = '#2ecc71';
  };

  document.getElementById('btn-reset').onclick = async () => {
    const info = document.getElementById('dash-info');
    const res = await google.colab.kernel.invokeFunction('notebook.ResetDataset', [], {});
    let data = res.data['application/json'] || JSON.parse(res.data['text/plain']?.replace(/'/g, '"') || '{}');
    if (data.counts) updateBadgeCounts(data.counts, 0);
    info.innerText = "Dataset has been cleared.";
    info.style.color = '#e67e22';
  };
}
initDatasetStudio();
"""

display(Javascript(DASHBOARD_JS))


## 1.2.1 Visual Inspection of 21-Joint Hand Skeletons (Dataset Inspection)

> **Quick Goal:** Visualize the 21-joint stick figure skeleton representations from your recorded dataset before training the Random Forest model.

* **Validating Feature Extraction:**
  * The computer does not see your face or skin tone; it only perceives **joint coordinates and connecting finger bones** as illustrated below.
  * The red dot at the bottom is the wrist anchor `(0, 0)`.
  * Notice how the index finger stands erect for `POINT`, two fingers separate for `PEACE`, and all five fingers curl inward for `FIST`.
* **Dataset Balance:**
  * The title of each box displays the number of recorded samples (`N`).
  * Ensure each gesture has a balanced sample count (e.g., at least 10–40 samples per gesture) so the AI learns fairly across all classes without bias.

In [ ]:
#@title 1.2.1 Visualize 21 Hand Skeletons in Dataset { display-mode: "form" }
# Displays 2D skeleton visualizations for each gesture class in the dataset

import matplotlib.pyplot as plt
import numpy as np

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),        # Index
    (0, 9), (9, 10), (10, 11), (11, 12),   # Middle
    (0, 13), (13, 14), (14, 15), (15, 16), # Ring
    (0, 17), (17, 18), (18, 19), (19, 20), # Pinky
    (5, 9), (9, 13), (13, 17), (0, 17)     # Palm
]

GESTURE_ORDER = [
    "POINT", "PEACE", "THREE_FINGERS", "FOUR_FINGERS",
    "OK_SIGN", "OPEN_PALM", "THUMBS_UP", "FIST"
]

if 'DATASET_X' not in globals() or len(DATASET_Y) == 0:
    print("[WARNING] Dataset is currently empty!")
    print("Please record samples in Cell 1.2 or click 'Load Synthetic Baseline' first.")
else:
    # Create 2-row x 4-column grid
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for idx, gesture_name in enumerate(GESTURE_ORDER):
        ax = axes[idx]
        
        # Find sample indices for this gesture
        sample_indices = [i for i, label in enumerate(DATASET_Y) if label == gesture_name]
        count = len(sample_indices)
        
        if count == 0:
            ax.text(0, 1.1, f"{gesture_name}\n(No Data Yet)", 
                    ha="center", va="center", color="gray", fontsize=11, fontweight="bold")
            ax.set_xlim(-1.6, 1.6)
            ax.set_ylim(-0.5, 3.0)
            ax.set_title(f"{gesture_name} (N = 0)", color="gray")
        else:
            sample_feats = np.array(DATASET_X[sample_indices[0]])
            pts = sample_feats.reshape(21, 2)
            
            # Coordinates x and y (invert y-axis so fingers point upwards)
            xs = pts[:, 0]
            ys = -pts[:, 1]
            
            # Draw finger bones (green lines)
            for p1, p2 in HAND_CONNECTIONS:
                ax.plot([xs[p1], xs[p2]], [ys[p1], ys[p2]], color="#27ae60", linewidth=2.5, zorder=2)
            
            # Draw joints (orange dots)
            ax.scatter(xs[1:20], ys[1:20], color="#e67e22", s=45, zorder=3)
            
            # Draw fingertips (golden yellow dots)
            tips = [4, 8, 12, 16, 20]
            ax.scatter(xs[tips], ys[tips], color="#f1c40f", s=75, edgecolors="#d35400", linewidth=1.5, zorder=4, label="Fingertips")
            
            # Draw wrist (red dot)
            ax.scatter(xs[0], ys[0], color="#e74c3c", s=90, edgecolors="#962d22", linewidth=1.5, zorder=5, label="Wrist (0,0)")
            
            ax.set_title(f"{gesture_name} (N = {count})", fontsize=12, fontweight="bold", pad=8)
            # Dynamic proportional axis limits so all hand joints and fingertips stay in view
            x_pad = max(float(np.abs(xs).max()), 1.2) + 0.35
            y_top = max(float(ys.max()) + 0.35, 2.8)
            y_bottom = min(float(ys.min()) - 0.25, -0.4)
            ax.set_xlim(-x_pad, x_pad)
            ax.set_ylim(y_bottom, y_top)
            ax.grid(True, linestyle="--", alpha=0.4)
        
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])

    plt.suptitle("MediaPipe 21-Joint Dataset Inspection (Stick Figures)", fontsize=16, fontweight="bold", y=0.98)
    plt.tight_layout()
    plt.show()
    print(f"Total dataset in memory: {len(DATASET_Y)} samples.")


## 1.3 Training the AI Brain: Random Forest Classifier

> **Quick Goal:** Teach the AI to read the 42-number finger joint tables and predict gesture names in milliseconds.

* **The 100 Junior Detectives Analogy (Random Forest):**
  * If we only ask a single friend to identify an object, they might make an honest mistake.
  * In a Random Forest, we recruit **100 junior detectives (Decision Trees)**.
  * Each detective checks 2–3 random clues (e.g., *"is the index finger extended?"*, *"is the thumb curled?"*).
  * At the end, all 100 detectives **cast a majority vote**. If 95 detectives shout "Thumbs Up!", the system is highly confident that the gesture is indeed a thumbs-up.
* **Why is this AI Architecture Ideal for Robotics?**
  * Training takes less than a second on a standard Colab CPU.
  * During live execution, it performs tens of predictions per second ($<2\text{ ms}$ inference) without requiring expensive GPU hardware.

### How It Works Behind the Scenes
* We split our dataset: 80% for learning (*Training Set*) and 20% for testing (*Test Set*).
* The model fits 100 voting trees, and the finalized model is serialized to `gesture_rf_model.joblib`.

> **Random Forest Ensemble Pipeline:**  
> `42 Joint Coordinates` ➔ `100 Decision Tree Detectives (Randomized Evaluation)` ➔ `Voting Ballot (Majority Vote)` ➔ `Final Output: Predicted Gesture`

> **Class Discussion Question:** *What would happen if we used only 1 decision tree instead of an ensemble? What risks arise when hands tremble or rotate slightly in the real world?*

In [ ]:
# --- DEFINE HYPERPARAMETERS HERE ---
n_trees = 100
max_tree_depth = 12
test_split_ratio = 0.20
random_seed = 42
# ---------------------------------------

import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

if len(DATASET_Y) < 16:
    print("[WARNING] Dataset is empty or has too few samples!")
    print("Please record samples in Cell 1.2 or click 'Load Synthetic Baseline'.")
else:
    X = np.array(DATASET_X)
    y = np.array(DATASET_Y)

    # Stratified Train and Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_split_ratio, random_state=random_seed, stratify=y
    )

    print(f"Training Random Forest on {len(X_train)} training samples...")
    rf_model = RandomForestClassifier(
        n_estimators=n_trees,
        max_depth=max_tree_depth,
        random_state=random_seed
    )
    rf_model.fit(X_train, y_train)

    # Evaluation on Test Data
    y_pred = rf_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Model Accuracy on Test Data: {acc * 100:.2f}%\n")

    print("Classification Performance Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    # Confusion Matrix Visualization
    cm = confusion_matrix(y_test, y_pred, labels=rf_model.classes_)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_model.classes_)
    disp.plot(ax=ax, cmap="Blues", colorbar=True, xticks_rotation=45)
    plt.title(f"Confusion Matrix Gesture Classifier (Accuracy: {acc*100:.1f}%)")
    plt.tight_layout()
    plt.show()

    # Save model for use in Chapter 3
    joblib.dump(rf_model, "gesture_rf_model.joblib")
    print("Model saved to 'gesture_rf_model.joblib'.")


### Reading the AI Report Card & The 100% Accuracy Pitfall

> **Quick Goal:** Interpret model accuracy using a Confusion Matrix and understand why a 100% score is not always completely bulletproof.

* **Reading the Evaluation Grid (Confusion Matrix):**
  * Numbers along the dark blue diagonal represent correct predictions.
  * Any numbers outside the diagonal indicate misclassifications (e.g., `PEACE` confused with `THREE_FINGERS`).
* **The 100% Trap (Overfitting):**
  * If your accuracy reaches 100%, do not celebrate prematurely!
  * If you only recorded samples with your hand locked in a single rigid pose, the AI merely memorized that exact photo rather than learning the generalized shape of the gesture.

## 1.4 Standalone Camera Testing (Offline / Without Connecting to Robot)

> **Quick Goal:** Test your live webcam to verify how reliably the AI recognizes your hand gestures before connecting to the physical robot arm.

* **Pre-Flight Test:**
  * Before operating an industrial machine weighing dozens of kilograms, we must verify our control signals safely in an offline environment.
  * Move your hand in front of your webcam and observe the predicted label box on the display.
* **The Flickering Phenomenon:**
  * Notice how predictions might flicker rapidly between classes when your fingers are mid-transition or slightly shaking.
  * **The Critical Bridge to Chapter 3:** This rapid flicker proves why we *never* directly wire raw camera output to robotic motors! In Chapter 3, we implement a **1-second Dwell Filter** to ensure the robot moves only when you hold an intentional, stable gesture.

> **Offline Verification Loop:**  
> `Laptop Webcam Frame` ➔ `Extract 21 Landmarks` ➔ `AI Prediction (<2 ms)` ➔ `Confidence Score Validation` ➔ `Live HUD Display`

> **Class Discussion Question:** *Try moving your hand very quickly across the camera frame. What happens to the predicted label? How can this issue be mitigated in a physical robotic control system?*

In [ ]:
#@title 1.4 Standalone Real-Time Camera Testing { display-mode: "form" }
# Please do not modify the contents of this cell
# 1.4 Prediction Functions & Real-Time Camera Testing (Offline / Without UR5)
import time
import asyncio
import cv2
import numpy as np
from IPython.display import HTML, Javascript, display
from google.colab.output import eval_js

def predict_gesture_with_confidence(feature_vector: np.ndarray):
    """Accepts 42 normalized features and returns predicted gesture name + confidence."""
    feats = feature_vector.reshape(1, -1)
    probs = rf_model.predict_proba(feats)[0]
    best_idx = np.argmax(probs)
    pred_label = rf_model.classes_[best_idx]
    confidence = float(probs[best_idx])
    return pred_label, confidence


OFFLINE_TESTER_JS = """
async function initOfflineTester() {
  const existing = document.getElementById('offline-box');
  if (existing) existing.remove();

  const container = document.createElement('div');
  container.id = 'offline-box';
  container.style.fontFamily = 'monospace';
  container.style.background = '#1a1a1a';
  container.style.color = '#fff';
  container.style.padding = '12px';
  container.style.borderRadius = '8px';
  container.style.display = 'inline-block';

  const header = document.createElement('div');
  header.innerHTML = '<b>Real-Time Gesture Model Testing (Offline / Without Robot)</b>';
  header.style.marginBottom = '6px';
  container.appendChild(header);

  const video = document.createElement('video');
  video.style.borderRadius = '6px';
  video.style.transform = 'scaleX(-1)';
  video.width = 400;
  video.height = 300;
  video.autoplay = true;
  video.muted = true;
  video.playsInline = true;
  container.appendChild(video);

  const hud = document.createElement('div');
  hud.id = 'offline-hud';
  hud.style.marginTop = '6px';
  hud.style.fontSize = '14px';
  hud.innerHTML = 'Status: Opening camera...';
  container.appendChild(hud);

  const btn = document.createElement('button');
  btn.innerText = 'Stop Testing';
  btn.style.marginTop = '8px';
  btn.style.background = '#e74c3c';
  btn.style.color = '#fff';
  btn.style.border = 'none';
  btn.style.padding = '6px 12px';
  btn.style.borderRadius = '4px';
  btn.style.cursor = 'pointer';

  btn.onclick = () => {
    window.offlineActive = false;
    if (window.offlineStream) window.offlineStream.getTracks().forEach(t => t.stop());
    btn.innerText = '⏹️ Testing Complete';
    btn.disabled = true;
    hud.innerHTML = 'Testing disabled.';
  };
  container.appendChild(btn);

  document.querySelector('#output-body').appendChild(container);

  const stream = await navigator.mediaDevices.getUserMedia({video: {width: 400, height: 300}});
  window.offlineStream = stream;
  video.srcObject = stream;
  await video.play();

  const canvas = document.createElement('canvas');
  canvas.width = 400;
  canvas.height = 300;
  const ctx = canvas.getContext('2d');

  window.offlineActive = true;
  window.captureOfflineFrame = function() {
    if (!window.offlineActive) return null;
    ctx.drawImage(video, 0, 0, 400, 300);
    return canvas.toDataURL('image/jpeg', 0.6);
  };

  window.updateOfflineHUD = function(label, conf) {
    const el = document.getElementById('offline-hud');
    if (!el || !window.offlineActive) return;
    const pct = Math.round(conf * 100);
    const color = conf >= 0.7 ? '#2ecc71' : '#e67e22';
    el.innerHTML = `Prediction: <b style="color:${color}; font-size:16px;">[${label}]</b> | Confidence: <b>${pct}%</b>`;
  };
}
initOfflineTester();
"""

async def test_gesture_model_offline(duration_seconds: float = 30.0):
    """Runs real-time inference visualization in browser without connecting to UR5 robot."""
    display(Javascript(OFFLINE_TESTER_JS))
    await asyncio.sleep(2.0)

    print("Offline camera testing active! Point your hand gestures toward the camera.")
    print(f"Testing will run automatically for {duration_seconds} seconds (or click the Stop button).\\n")

    t_start = time.time()
    try:
        while (time.time() - t_start) < duration_seconds:
            is_active = eval_js('window.offlineActive ? true : false')
            if not is_active:
                break

            frame_b64 = eval_js('window.captureOfflineFrame ? window.captureOfflineFrame() : null')
            if frame_b64 is None:
                await asyncio.sleep(0.05)
                continue

            header, encoded = frame_b64.split(',', 1) if ',' in frame_b64 else ('', frame_b64)
            img_bytes = base64.b64decode(encoded)
            img_np = np.frombuffer(img_bytes, dtype=np.uint8)
            frame = cv2.imdecode(img_np, flags=cv2.IMREAD_COLOR)
            rgb = cv2.cvtColor(cv2.flip(frame, 1), cv2.COLOR_BGR2RGB)

            lms = hands_detector.detect_landmarks(rgb)
            if lms is not None:
                feats = extract_normalized_features(lms)
                if feats is not None:
                    label, conf = predict_gesture_with_confidence(feats)
                    eval_js(f'window.updateOfflineHUD ? window.updateOfflineHUD("{label}", {conf}) : null')
            else:
                eval_js('window.updateOfflineHUD ? window.updateOfflineHUD("NO HAND DETECTED", 0.0) : null')

            await asyncio.sleep(0.04)

    finally:
        eval_js('if (window.offlineStream) { window.offlineStream.getTracks().forEach(t => t.stop()); window.offlineActive = false; }')
        print("Offline testing complete.")


# Run offline camera testing (30 seconds)
await test_gesture_model_offline(duration_seconds=30.0)


# Chapter 2: Large Language Models with Google AI Studio

In Chapter 1 we built **perception**: camera → hand landmarks → features → gesture labels. Chapter 2 builds **planning**: a natural-language sentence → a model answer. We use Google AI Studio here so every student can run the same cells on their own API key; the UR5 is still actuated by the lab server in Chapter 3.

| Section | What you will do | Cells |
| --- | --- | --- |
| 2.1 | Set up the Gemini API key | 2.1 |
| 2.2 | Make a first call with the Python SDK | 2.2 |
| 2.3 | Build a chatbot with a model selector | 2.3.1–2.3.2 |
| 2.4 | Change behaviour with a system prompt | 2.4.1–2.4.2 |

Two rules never change:

1. **The key stays in the kernel.** Type it once in cell 2.1; it is exported to `GEMINI_API_KEY` and never written into the notebook.
2. **The LLM only plans.** Guardrails, workspace limits, and hardware execution still belong to the lab server in Chapter 3.

> Tested responsive models (Nov 2026): `gemini-3.5-flash-lite`, `gemini-3.1-flash-lite`, `gemini-flash-lite-latest`, `gemini-3-flash-preview`, `gemini-3.6-flash`. Busy at test time: `gemini-3.8-flash`, `gemma-4-31b-it` (HTTP 503).

---


## 2.1 Getting a Gemini API Key

> **Quick Goal:** Create a free Google AI Studio API key and paste it into the password box in cell 2.1.

* **Steps:**
  1. Open **Google AI Studio → API Keys**: https://aistudio.google.com/api-keys (sign in with your Google account).
  2. Click **Create API key** (top right). If asked, choose **Create API key in new project**.
  3. The new key appears in the **Key** column, grouped under the **Default Gemini Project** (or your chosen project).
  4. Hover over the key row and click the **copy** icon — the tooltip reads **Copy API key** (see the diagram below).
  5. Paste the copied key into the input box in cell 2.1. It is exported to `GEMINI_API_KEY` immediately as you type.

> Diagram rendered from Mermaid via `mermaid.ink`, so it loads in Colab without a local image file. The original screenshot is kept at `assets/gemini_api_keys.png` in the repo for reference.

> **Keep it private:** never commit the key to a repository or share it with classmates. The **Free tier** is enough for this workshop. Check your remaining quota at **Rate Limit** in the left sidebar.

---


In [ ]:
#@title 2.1 Setup — Gemini API Key { display-mode: "form" }
# Run this cell, then paste your API key into the box below.
# The key is exported to GEMINI_API_KEY as you type - no second cell needed.

import os
import ipywidgets as widgets
from IPython.display import display


def _export_key(change):
    os.environ["GEMINI_API_KEY"] = change["new"].strip()


gemini_api_key = widgets.Password(
    value="",
    placeholder="Paste your Google AI Studio API key here",
    description="API key:",
    layout=widgets.Layout(width="640px"),
)
gemini_api_key.observe(_export_key, names="value")

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;color:#89b4fa;'>SETUP / GEMINI API KEY</h3>"),
    gemini_api_key,
    widgets.HTML(
        "<span style='color:#868e96;font-size:13px;'>"
        "Type your key above. It stays in kernel memory only and is exported "
        "to GEMINI_API_KEY immediately - it is never written to the notebook.</span>"
    ),
]))
print("Setup ready. Paste your key above, then run cell 2.2.")


## 2.2 First Call with the Python SDK

> **Quick Goal:** Verify that the exported key works by making one text-generation call with the official Google AI Studio SDK.

* **Why an SDK instead of raw HTTP?**
  * The `google-genai` client handles authentication, request formatting, and response parsing for you.
  * `genai.Client()` automatically reads `GEMINI_API_KEY` from the environment — the same variable cell 2.1 just set.
  * `client.interactions.create(...)` is the simplest documented entry point: one `model` and one `input`.
* **Choosing a Model:**
  * `gemini-3.5-flash-lite` is fast and lightly loaded, so it is used as the default.
  * The docs example uses `gemini-4-hydrogen`, which may return HTTP 503 during high demand.

### How It Works Behind the Scenes

* The SDK is installed once in cell 0.1 with pinned versions (`google-genai==2.12.1`), so Colab's own dependencies stay intact.
* `interaction.output_text` holds only the generated text; the full response also carries usage metadata.

> **SDK Call Pipeline:**  
> `GEMINI_API_KEY (env)` ➔ `genai.Client()` ➔ `interactions.create(model, input)` ➔ `interaction.output_text`

> **Class Discussion Question:** *What would change if you swapped `gemini-3.5-flash-lite` for a model that returns HTTP 503? Where would you see the error?*


In [ ]:
#@title 2.2 First Call with the Python SDK { display-mode: "form" }
# The API key was exported by cell 2.1. Make one call with genai.Client().
# The SDK itself is installed in cell 0.1 with pinned versions.

import os
from google import genai

print("GEMINI_API_KEY loaded:", bool(os.environ.get("GEMINI_API_KEY")))

client = genai.Client()
interaction = client.interactions.create(
    model="gemini-3.5-flash-lite",   # docs example: gemini-4-hydrogen (often 503)
    input="Explain how AI works in a few words",
)
print(interaction.output_text)


## 2.3 Chatbot

> **Quick Goal:** Build a chatbot that streams answers from Gemini and displays the whole conversation as a chat transcript.

* **Two Code Cells:**
  * **2.3.1 Input** — prepare the model dropdown and prompt box. Five tested models are offered; the selection is read at run time by 2.3.2, so you can switch models without re-running the input cell.
  * **2.3.2 Runner Display** — send the prompt, stream the answer, and render every turn.
* **No Key Field:** the key is already exported by cell 2.1; repeating it in every cell would be redundant and risky.
* **Full Transcript:** on every run the runner first renders all of `ch2_history`, then appends the new user turn and the model answer.
* **Live Streaming:** native Gemini streaming (`streamGenerateContent?alt=sse`) emits `data:` chunks; each chunk updates only the last bubble, so the panel does not flicker.
* **Auto-Scroll:** a `MutationObserver` in the browser scrolls the panel to the bottom whenever the content changes.

### How It Works Behind the Scenes

* ipywidgets keeps the controls live; `gemini_model` and `gemini_input` hold the current widget objects.
* Gemini uses `contents[{role, parts:[{text}]}]` (roles `user` / `model`) instead of OpenAI's `messages`.
* The API key is read from `GEMINI_API_KEY`, exported by cell 2.1; the runner never stores it in a widget.
* HTTP errors (401, 429, 503) appear inside the assistant bubble instead of stopping the cell.

> **Chatbot Pipeline:**  
> `2.3.1 Input (model + prompt)` ➔ `contents[]` ➔ `streamGenerateContent?alt=sse` ➔ `SSE chunks` ➔ `Chat Bubbles` ➔ `Auto-Scroll`

> **Class Discussion Question:** *Why does the runner update a single bubble per chunk instead of creating a new bubble for every chunk?*

---


In [ ]:
#@title 2.3.1 Chatbot — Input { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display

GEMINI_MODELS = [
    ("gemini-3.5-flash-lite", "gemini-3.5-flash-lite"),
    ("gemini-3.1-flash-lite", "gemini-3.1-flash-lite"),
    ("gemini-flash-lite-latest", "gemini-flash-lite-latest"),
    ("gemini-3-flash-preview", "gemini-3-flash-preview"),
    ("gemini-3.6-flash", "gemini-3.6-flash"),
]

# The API key is set up once in cell 2.1. No key field here.
gemini_model = widgets.Dropdown(
    options=GEMINI_MODELS, value="gemini-3.5-flash-lite", description="Model:",
)
gemini_input = widgets.Text(
    value="Explain in one sentence: what is a robot planner?",
    description="Chat:", layout=widgets.Layout(width="560px"),
)

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;color:#89b4fa;'>GEMINI CHAT / AI STUDIO</h3>"),
    gemini_model, gemini_input,
    widgets.HTML(
        "<span style='color:#868e96;font-size:13px;'>"
        "Key from cell 2.1. Switch the model any time, then run cell 2.3.2.</span>"
    ),
]))
print("Gemini chat form ready. Run cell 2.3.2.")


In [ ]:
#@title 2.3.2 Chatbot — Runner Display { display-mode: "form" }
# Run AFTER cell 2.3.1. Renders full history, then streams the new turn.

import html
import json
import time
import urllib.request
import urllib.error
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output as _colab_output
except Exception:
    _colab_output = None

if "gemini_input" not in globals() or "gemini_model" not in globals():
    raise RuntimeError("Run cell 2.3.1 before cell 2.3.2.")

if "ch2_history" not in globals():
    ch2_history = []

API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()  # exported by cell 2.1
MODEL = gemini_model.value
user = gemini_input.value.strip()

_SCROLL_ID = f"gemscroll-{int(time.time() * 1000)}"
_STYLE = ("border:1px solid #454a55;border-radius:8px;padding:10px;"
          "background:#11151c;max-height:420px;overflow:auto;")
panel = widgets.HTML(value=f"<div id='{_SCROLL_ID}' style='{_STYLE}'></div>")
_bubbles = []


def _bubble(role, text, streaming=False):
    label = "USER" if role == "user" else "MODEL"
    color = "#89b4fa" if role == "user" else "#a6e3a1"
    align = "flex-end" if role == "user" else "flex-start"
    bg = "#233049" if role == "user" else "#1c2422"
    body = html.escape(text or "").replace("\n", "<br>")
    if streaming:
        body += " ▌"
    if not body.strip():
        body = "<span style='color:#6c7086;'>Waiting for answer...</span>"
    return (
        f"<div style='display:flex;justify-content:{align};margin:8px 0;'>"
        f"<div style='max-width:82%;'>"
        f"<div style='color:{color};font-size:11px;font-family:monospace;margin-bottom:3px;'>{label}</div>"
        f"<div style='border:1px solid #454a55;border-radius:8px;padding:8px 10px;"
        f"background:{bg};color:#cdd6f4;font-family:monospace;font-size:12px;"
        f"line-height:1.6;white-space:pre-wrap;'>{body}</div></div></div>"
    )


def _paint():
    panel.value = f"<div id='{_SCROLL_ID}' style='{_STYLE}'>" + "".join(_bubbles) + "</div>"


def _append(role, text="", streaming=False):
    _bubbles.append(_bubble(role, text, streaming))
    _paint()


def _update_last(text, streaming=True):
    if _bubbles:
        _bubbles[-1] = _bubble("model", text, streaming)
        _paint()


display(widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;color:#89b4fa;'>GEMINI CHAT / AI STUDIO</h3>"),
    panel,
]))

if _colab_output is not None:
    _colab_output.eval_js(f"""
new Promise(resolve => {{
  const el = document.getElementById('{_SCROLL_ID}');
  const host = el && el.parentElement;
  if (!host) {{ resolve('missing'); return; }}
  new MutationObserver(() => {{
    const cur = document.getElementById('{_SCROLL_ID}');
    if (cur) cur.scrollTop = cur.scrollHeight;
  }}).observe(host, {{childList: true, subtree: true, characterData: true}});
  resolve('attached');
}})
""")

for m in list(ch2_history):
    _append(m["role"], m["content"])

if not API_KEY:
    _append("model", "API key is empty. Run cell 2.1 and paste your key there.")
elif not user:
    _append("model", "Prompt is empty. Fill it in cell 2.3.1.")
else:
    ch2_history.append({"role": "user", "content": user})
    _append("user", user)
    _append("model")

    contents = []
    for m in ch2_history:
        role = "user" if m["role"] == "user" else "model"
        contents.append({"role": role, "parts": [{"text": m["content"]}]})

    url = (f"https://generativelanguage.googleapis.com/v1beta/models/"
           f"{MODEL}:streamGenerateContent?alt=sse&key={API_KEY}")
    req = urllib.request.Request(
        url,
        data=json.dumps({"contents": contents}).encode(),
        headers={"Content-Type": "application/json"},
    )

    _buf = [""]
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            for line in resp:
                line = line.strip()
                if not line.startswith(b"data:"):
                    continue
                chunk = json.loads(line[5:])
                piece = ""
                for cand in chunk.get("candidates") or []:
                    for part in (cand.get("content") or {}).get("parts") or []:
                        piece += part.get("text") or ""
                if piece:
                    _buf[0] += piece
                    _update_last(_buf[0], streaming=True)
    except urllib.error.HTTPError as exc:
        _buf[0] += f"\nHTTP {exc.code}: {exc.read().decode()[:250]}"
        _update_last(_buf[0], streaming=False)
    except Exception as exc:
        _buf[0] += f"\nERROR: {exc}"
        _update_last(_buf[0], streaming=False)
    else:
        answer = _buf[0]
        _update_last(answer, streaming=False)
        ch2_history.append({"role": "model", "content": answer})
        print(f"DONE | model={MODEL} | {len(ch2_history)} messages in history.")


## 2.4 System Prompt

> **Quick Goal:** Understand what a system prompt does, how it differs from the user prompt, and see how one rule changes the answer.

### What Is a System Prompt?

The **system prompt** is the rule layer of a request. It tells the model what role to play, how to behave, and what an acceptable answer looks like. It applies to the **whole conversation**, not just one message.

**Function**
* Sets the role and goal, e.g. "You are an action planner for a UR5 robot arm".
* Fixes the output format and vocabulary: one action + one chess square, no explanation, no Markdown.
* Defines consistency rules that must hold for every turn, so the model behaves like a planner instead of a chatty assistant.
* This is the closest thing to "programming" an LLM without retraining it.

### System Prompt vs User Prompt

| | System prompt | User prompt |
| --- | --- | --- |
| **Purpose** | Rules, role, constraints | Data, world state, the actual task |
| **Example** | "Choose exactly ONE action from: pick, place, move, move to home…" | "Item positions: item_1=a1, item_2=h8. Task: move item_1 to c3." |
| **Where it goes** | `systemInstruction` — hidden from the chat transcript | `contents[].parts[].text` — visible as a user message |
| **When it is sent** | Once per request, applies to every turn | Every turn, appended to the conversation history |
| **Priority** | Higher — the model should follow it even if the user says otherwise | Lower — it must respect the system rules |

> Rule of thumb: **rules before data**. Put the rules in the system prompt; put the situation and the question in the user prompt.

### How to Use It in This Section

* **Two Code Cells:**
  * **2.4.1 Input** — write the **SYSTEM INSTRUCTION** (rules) in `sys_box` and the **USER MESSAGE** (world state + task) in `user_box`.
  * **2.4.2 Runner Display** — send both fields and show the answer in a compact panel.
* The call uses `generateContent` (one shot, no streaming) because the expected answer is one short action; the key is passed as `?key=`.
* The model selector from 2.3.1 is reused, so you can compare how different models obey the same rule.
* `usageMetadata` in the response shows token counts, useful when comparing prompt styles.
* **Try it:** change one rule in 2.4.1 while keeping the user message fixed, then run 2.4.2 and see which models still return `pick a1`.

> **System Prompt Pipeline:**  
> `2.4.1 Input` ➔ `systemInstruction + contents[]` ➔ `generateContent?key=...` ➔ `candidates[0].parts[0].text` ➔ `HTML Panel`

> **Class Discussion Question:** *If the model ignores the action list, which part of the request would you inspect first?*

---


In [ ]:
#@title 2.4.1 System Prompt — Input { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display

if "gemini_model" not in globals():
    raise RuntimeError("Run cell 2.3.1 before cell 2.4.1.")

sys_box = widgets.Textarea(
    value=(
        "You are an action planner for a UR5 robot arm with a Robotiq gripper.\n"
        "Choose exactly ONE action from: pick, place, move, move to home, task_completed, reject_task.\n"
        "Squares use chess notation a1 to h8.\n"
        "Reply with the action and square only, e.g. 'pick a1'. No explanation, no markdown."
    ),
    layout=widgets.Layout(width="640px", height="120px"),
)
user_box = widgets.Textarea(
    value=(
        "Item positions: item_1=a1, item_2=h8. Gripper empty.\n"
        "Task: move item_1 to c3."
    ),
    layout=widgets.Layout(width="640px", height="80px"),
)
display(widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;color:#89b4fa;'>SYSTEM PROMPT</h3>"),
    widgets.HTML("<b style='color:#f9e2af;font-size:12px;'>SYSTEM INSTRUCTION</b>"),
    sys_box,
    widgets.HTML("<b style='color:#f9e2af;font-size:12px;'>USER MESSAGE</b>"),
    user_box,
    widgets.HTML(
        "<span style='color:#868e96;font-size:13px;'>"
        "Uses the key from cell 2.1 and the model selected in 2.3.1. Then run cell 2.4.2.</span>"
    ),
]))
print("System prompt form ready. Run cell 2.4.2.")


In [ ]:
#@title 2.4.2 System Prompt — Runner Display { display-mode: "form" }
import html
import json
import urllib.request
import urllib.error
import ipywidgets as widgets
from IPython.display import display

import os

if "sys_box" not in globals():
    raise RuntimeError("Run cell 2.4.1 before cell 2.4.2.")

key = os.environ.get("GEMINI_API_KEY", "").strip()  # exported by cell 2.1
model = gemini_model.value
sys_text = sys_box.value.strip()
user_text = user_box.value.strip()

panel2 = widgets.HTML(value="")
display(panel2)

if not key:
    panel2.value = "<div style='color:#f38ba8;'>API key is empty. Run cell 2.1 and paste your key there.</div>"
elif not sys_text or not user_text:
    panel2.value = "<div style='color:#f9e2af;'>Fill both fields in cell 2.4.1.</div>"
else:
    body = {
        "systemInstruction": {"parts": [{"text": sys_text}]},
        "contents": [{"role": "user", "parts": [{"text": user_text}]}],
    }
    url = (f"https://generativelanguage.googleapis.com/v1beta/models/"
           f"{model}:generateContent?key={key}")
    req = urllib.request.Request(
        url, data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.load(resp)
        answer = ""
        for cand in data.get("candidates") or []:
            for part in (cand.get("content") or {}).get("parts") or []:
                answer += part.get("text") or ""
        panel2.value = (
            "<div style='border:1px solid #454a55;border-radius:8px;padding:10px;"
            "background:#11151c;color:#cdd6f4;font-family:monospace;font-size:12px;'>"
            + html.escape(answer).replace("\n", "<br>") + "</div>"
        )
        print(f"DONE | model={model}")
    except urllib.error.HTTPError as exc:
        panel2.value = f"<div style='color:#f38ba8;'>HTTP {exc.code}: {html.escape(exc.read().decode()[:250])}</div>"
    except Exception as exc:
        panel2.value = f"<div style='color:#f38ba8;'>ERROR: {html.escape(str(exc))}</div>"


### Chapter 2 Summary

| Concept | Notebook Cell | Observed Result |
| --- | --- | --- |
| API key setup | cell 2.1 | `GEMINI_API_KEY` exported from the input box |
| Python SDK | cell 2.2 | `genai.Client()` → `interaction.output_text` |
| Chatbot | cells 2.3.1–2.3.2 | Full transcript, streamed chunks, model selector |
| System prompt | cells 2.4.1–2.4.2 | One rule changes the answer |

### Pre-Lab Checklist

1. Cell 2.1: paste your key; cell 2.2 prints `GEMINI_API_KEY loaded: True`.
2. Cell 2.3.2: switch the model between runs and confirm the answer still arrives.
3. Cell 2.4.1: edit the system instruction and observe the change in cell 2.4.2.
4. Continue to Chapter 3 to connect planning to the physical UR5 through `session`.


# Chapter 3: Hands-on Learning with the UR5 Robotic Manipulator

In Chapter 3, we connect our programs to the real industrial UR5 robot arm in the laboratory. This module is built with modular flexibility, supporting both hand gesture teleoperation and automated action sequences.

---

## 3.1 Lab Network Architecture & Time-Leased Mutex Lock

> **Quick Goal:** Establish a secure communication channel from Google Colab to the lab workstation and understand collaborative queue management to prevent conflicting robot commands.

* **The Cloudflare Tunnel Bridge:**
  * Google Colab runs in Google's cloud infrastructure, while the physical UR5 robot sits on a laboratory workbench behind a local university firewall.
  * We utilize a secure **Cloudflare Tunnel** (WebSocket over SSL / WSS) as an encrypted virtual gateway, eliminating the need for public IP addresses or port forwarding.
* **The Restroom Key with an Egg Timer Analogy (Time-Leased Mutex Lock):**
  * There is only **1 physical robot arm** in the lab, but **dozens of students** across multiple groups operating laptops simultaneously.
  * If multiple participants trigger movements simultaneously, the robot would receive conflicting trajectories, trip safety limits, and trigger an emergency stop.
  * The lab server employs an automated digital lock: any group wishing to actuate the arm borrows a temporary control token (lease) for 30–60 seconds.
  * If your network connection drops while holding the lease, an internal timeout expires and automatically reclaims the token, preventing the robot from becoming permanently locked.

### How It Works Behind the Scenes
* The `RobotSession` client establishes a full-duplex asynchronous WebSocket connection.
* Any motion request is strictly rejected by the server unless accompanied by a valid, active lease token.

> **Communication & Leasing Pipeline:**  
> `Colab Notebook Client` ➔ `Cloudflare Tunnel (WSS)` ➔ `Lab Server Mutex Lock (Lease Timer)` ➔ `Physical UR5 Motion`

> **Class Discussion Question:** *Why must the control lease automatically expire upon inactivity rather than relying on manual release by the user?*

In [ ]:
#@title Robot Session Client Module (RobotSession) { display-mode: "form" }
# Please do not modify the contents of this cell
import asyncio
import json
import time
import websockets
def _msg(msg_type: str, payload: dict) -> str:
    """Standard envelope serializer helper for robot_server communication."""
    return json.dumps({
        "msg_id": str(int(time.time() * 1000) % 10**8),
        "msg_type": msg_type,
        "timestamp": time.time(),
        "payload": payload,
    })
class RobotSession:
    """
    Asynchronous communication client to Lab PC server.
    Manages mutex lock leasing lifecycle, quadrant sync, and robot motion execution.
    """
    def __init__(self, ws_url: str, student_id: str):
        self.ws_url = ws_url
        self.student_id = student_id
        self.token = None
        self.quadrant_config = None
    def set_quadrant_config(self, config: dict):
        """Saves active quadrant configuration to synchronize with lab server."""
        self.quadrant_config = dict(config)
        print(f"[SESSION] Quadrant configuration synchronized: {self.quadrant_config}")
    async def ping(self) -> dict:
        """Sends heartbeat message to test server connectivity."""
        try:
            async with websockets.connect(self.ws_url, open_timeout=5.0) as ws:
                await ws.send(_msg("HEARTBEAT", {"client": self.student_id}))
                raw = await asyncio.wait_for(ws.recv(), timeout=5.0)
                return json.loads(raw).get("payload", {})
        except Exception as e:
            return {"error": str(e)}
    async def acquire_lock(self, duration_s: float = 45.0) -> bool:
        """Requests an exclusive control lease token for the robot."""
        try:
            async with websockets.connect(self.ws_url, open_timeout=5.0) as ws:
                req = _msg("ACQUIRE_LEASE", {"student_id": self.student_id, "duration_s": duration_s})
                await ws.send(req)
                raw = await asyncio.wait_for(ws.recv(), timeout=5.0)
                data = json.loads(raw).get("payload", {})
                if data.get("granted"):
                    self.token = data.get("token")
                    print(f"[LEASING] Control lease successfully acquired for '{self.student_id}'. Active for {duration_s} seconds.")
                    return True
                else:
                    holder = data.get("holder", "another group")
                    wait_time = data.get("wait_seconds", "?")
                    print(f"[LEASING] Robot is currently in use by '{holder}'. Remaining lease time: {wait_time} seconds.")
                    return False
        except Exception as e:
            print(f"[LEASING ERROR] Failed to reach lease manager server: {e}")
            return False
    async def release_lock(self) -> bool:
        """Releases control lease token so the robot can be used by other groups."""
        if not self.token:
            return True
        try:
            async with websockets.connect(self.ws_url, open_timeout=5.0) as ws:
                req = _msg("RELEASE_LEASE", {"student_id": self.student_id, "token": self.token})
                await ws.send(req)
                raw = await asyncio.wait_for(ws.recv(), timeout=5.0)
                data = json.loads(raw).get("payload", {})
                self.token = None
                print(f"[LEASING] Control lease for '{self.student_id}' safely released.")
                return data.get("released", True)
        except Exception as e:
            print(f"[LEASING ERROR] An issue occurred while releasing lease: {e}")
            self.token = None
            return False
    async def execute(self, action: str, quadrant_config: dict = None) -> bool:
        """Sends single action command to physical robot/simulator along with active quadrant config."""
        plan_id = f"plan_{int(time.time() * 1000) % 10**6}"
        cfg = quadrant_config or self.quadrant_config
        params = {"token": self.token, "client": self.student_id}
        if cfg:
            params["quadrant_config"] = cfg
        payload = {
            "plan_id": plan_id,
            "actions": [action],
            "parameters": params,
        }
        try:
            async with websockets.connect(self.ws_url, open_timeout=10.0) as ws:
                await ws.send(_msg("PLAN_DISPATCH", payload))
                ack_raw = await asyncio.wait_for(ws.recv(), timeout=8.0)
                ack = json.loads(ack_raw).get("payload", {})
                if not ack.get("accepted"):
                    print(f"[DISPATCH REJECTED] {ack.get('message', 'Command not accepted by server')}")
                    return False
                print(f"[STATUS: RUNNING] Executing command [{action}]...")
                while True:
                    raw = await asyncio.wait_for(ws.recv(), timeout=35.0)
                    msg = json.loads(raw)
                    mtype = msg.get("msg_type")
                    p = msg.get("payload", {})
                    if mtype == "STEP_PROGRESS":
                        if p.get("status") == "SUCCESS":
                            elapsed = p.get("elapsed_s", 0.0)
                            print(f"[STATUS: SUCCESS] Completed in {elapsed:.2f} seconds.")
                        elif p.get("status") == "FAILED":
                            print(f"[STATUS: FAILED] Step execution failed: {p.get('message')}")
                            return False
                    elif mtype == "PLAN_COMPLETE":
                        status = p.get("status")
                        return status == "COMPLETED"
        except Exception as e:
            print(f"[EXECUTION ERROR] Connection lost while executing [{action}]: {e}")
            return False
print("RobotSession client module ready to use.")


## 3.2 Group Identity and Server URL Configuration

> **Quick Goal:** Set a unique group identifier so your control lease on the lab server is never confused with other participant teams.

* **Unique Group Name:** Replace `Group_01` with your team's designated ID (e.g., `Group_03`).
* **Why Must It Be Unique?** If two groups submit identical IDs, the server considers them the same client, causing race conditions for the active lease token!
* **Server Address:** Verify that the WebSocket tunnel address (`wss://ur5.fawzqi.my.id`) matches the instructor's projection screen.

### How It Works Behind the Scenes
* The Colab Form enables clean parameter input without modifying raw Python code. The instantiated `session` object manages network communication in memory.

In [ ]:
#@title Robot Control Session Configuration Form { display-mode: "form" }
# --- CONFIGURE GROUP IDENTITY HERE ---
STUDENT_ID = "Group_01"  #@param {type:"string"}
ROBOT_TUNNEL_WS = "wss://ur5.fawzqi.my.id"  #@param {type:"string"}
# -------------------------------------------
session = RobotSession(ws_url=ROBOT_TUNNEL_WS, student_id=STUDENT_ID)
print(f"Session client initialized for: {STUDENT_ID}")
print(f"Target Server: {ROBOT_TUNNEL_WS}")


## 3.3 Initial Handshake & HOME_POS Benchmark Test

> **Quick Goal:** Test network connectivity, acquire a temporary lease token, and command the robot to transit to the safe `HOME_POS` benchmark.

* **3 Safe Verification Steps:**
  * **Step 1 (Ping Heartbeat):** Verify that the network tunnel between Colab and the Lab PC is responsive.
  * **Step 2 (Acquire Lease):** Request an exclusive hardware control token from the server.
  * **Step 3 (Benchmark Motion):** Command the robot arm to transit to the pre-calibrated `HOME_POS` coordinate.
* **The `HOME_POS` Benchmark:** An upright, safe rest configuration verified by lab instructors to confirm that Colab commands successfully actuate physical joint motors.

### How It Works Behind the Scenes
* `verify_initial_connection()` sends a ping heartbeat, acquires a temporary lease, executes `HOME_POS`, and cleanly releases the token upon completion.

> **Handshake Verification Flow:**  
> `1. Ping Heartbeat` ➔ `2. Acquire Lease (30s)` ➔ `3. Transit to HOME_POS Benchmark` ➔ `4. Release Lease Cleanly`

> **Class Discussion Question:** *If the ping test succeeds but the lease request is denied, is the network connection broken? What is actually happening on the server?*

In [ ]:
#@title Execute Handshake Test & HOME_POS Benchmark { display-mode: "form" }
async def verify_initial_connection():
    print("[1/3] Sending ping signal to lab server...")
    status = await session.ping()
    if "error" in status:
        print(f"[WARNING] Failed to connect to server: {status['error']}")
        print("Please ensure the server and Cloudflare tunnel on the lab PC are active.")
        return False
    print(f"[2/3] Successfully connected. Server lease status: {status.get('lease_status')}")
    
    print(f"[3/3] Requesting control lease for initial benchmark test...")
    has_lock = await session.acquire_lock(duration_s=30.0)
    if not has_lock:
        print("Lock is currently held by another group. Please try again in a few moments.")
        return False
    print("Sending benchmark test command: [HOME_POS] from captured_points.json...")
    ok_home = await session.execute("HOME_POS")
    if ok_home:
        print("Benchmark point [HOME_POS] executed successfully!")
    await session.release_lock()
    
    if ok_home:
        print("Chapter 3 initial test successful! Robot session, waypoint database, and network are ready.")
    return ok_home
await verify_initial_connection()


### Evaluation of Handshake Test 3.3: Three Verified Milestones

> **Quick Goal:** Confirm all foundational subsystems are functional before moving to advanced teleoperation.

* **1. Secure Network Tunnel Active:** Instruction packets successfully traversed the Cloudflare tunnel to the campus lab computer.
* **2. Concurrency Control Verified:** The digital mutex token safely prevented simultaneous multi-user hardware contention.
* **3. Hardware Actuation Confirmed:** The lab controller decoded waypoint coordinates and actuated the physical robot joints smoothly.

## 3.4 Chessboard Workspace Grid (A1 - H8) & Action Mapping

> **Quick Goal:** Discretize the robot workspace using an $8 \times 8$ chessboard coordinate system and map hand gestures to target quadrants.

* **The Chessboard & Battleship Analogy:**
  * Instead of telling the robot *"move 14.3 cm forward and 8.7 cm left"*, we place a standard $8 \times 8$ chessboard in front of the robot.
  * We simply command: *"Go to square B2!"* The robot controller references pre-calibrated Cartesian coordinates.
* **4 Table Quadrants:**
  * `Q1` (Front-Left) : Square `B2`
  * `Q2` (Front-Right) : Square `G2`
  * `Q3` (Back-Left) : Square `G7`
  * `Q4` (Back-Right) : Square `B7`
  * `BOX` (Placement Bin) : Calibrated drop-off container beside the workspace table.
* **Customizable Squares:** You may modify quadrant square assignments in the code (e.g., setting `Q1` to `C4`). The server validates chessboard bounds automatically.

> **Chessboard Grid & Quadrant Layout:**  
> `[Q3: Back-Left (G7)]` ── `[Q4: Back-Right (B7)]`  
> `[Q1: Front-Left (B2)]` ── `[Q2: Front-Right (G2)]` ➔ `[Drop-off Container: BOX]`

> **Class Discussion Question:** *Why is guiding a robot via discrete chessboard squares (B2, G2) significantly safer than unconstrained continuous fingertip tracing in 3D space?*

In [ ]:
#@title Chessboard Quadrant Configuration & Action Mapping { display-mode: "form" }
# --- CONFIGURE QUADRANTS HERE ---
# 3.4 Chessboard Quadrant Configuration & Gesture Action Mapping
# Students can customize target squares on the chessboard (A1 - H8) and the drop-off container BOX.
QUADRANT_CONFIG = {
    "Q1": "B2",   # Quadrant 1 (A1-D4)
    "Q2": "G2",   # Quadrant 2 (E1-H4)
    "Q3": "G7",   # Quadrant 3 (E5-H8)
    "Q4": "B7",   # Quadrant 4 (A5-D8)
    "BIN": "BOX", # Drop-off container
}
# Mapping from Machine Learning predictions to robot actions
GESTURE_TO_ACTION = {
    "POINT": "HOVER_Q1",         # 1 Finger -> Hover Quadrant 1
    "PEACE": "HOVER_Q2",         # 2 Fingers -> Hover Quadrant 2
    "THREE_FINGERS": "HOVER_Q3", # 3 Fingers -> Hover Quadrant 3
    "FOUR_FINGERS": "HOVER_Q4",  # 4 Fingers -> Hover Quadrant 4
    "OK_SIGN": "PICK_SEQUENCE",  # OK Sign -> Pick Sequence (Hover -> Open -> Descend -> Grip -> Lift)
    "OPEN_PALM": "RELEASE_BIN",  # Open Palm -> Place Sequence (To BOX -> Descend -> Open -> Lift)
    "THUMBS_UP": "READY",        # Thumb -> Neutral / Ready Position
    "FIST": None,                # Fist -> Hold / Neutral (Safety Deadzone)
}
# ── Validate Quadrant Configuration ──
def validate_quadrant_config(config: dict):
    """Ensures each chessboard square is valid before dispatching plan to physical robot."""
    valid_files = set("ABCDEFGH")
    valid_ranks = set("12345678")
    valid_squares = {f"{f}{r}" for f in valid_files for r in valid_ranks} | {"BOX"}
    required_keys = ["Q1", "Q2", "Q3", "Q4", "BIN"]
    for k in required_keys:
        if k not in config:
            raise KeyError(f"Key '{k}' not found in QUADRANT_CONFIG! Must define Q1, Q2, Q3, Q4, and BIN.")
        
        sq = str(config[k]).strip().upper()
        if sq not in valid_squares:
            raise ValueError(
                f"Chessboard square '{sq}' in [{k}] is INVALID!\n"
                f"   Square must be within range A1 - H8 (e.g., 'B2', 'G7') or 'BOX'."
            )
    # Synchronize configuration with session object to automatically dispatch to lab server
    if "session" in globals() and hasattr(session, "set_quadrant_config"):
        session.set_quadrant_config(config)
    print("Validation Successful! All target squares are valid:")
    for q_key, sq in config.items():
        print(f"   • {q_key:<4} -> Square [{sq}]")
# Run validation
validate_quadrant_config(QUADRANT_CONFIG)


## 3.5 Dwell Debouncer Filter & Motion Idempotency

> **Quick Goal:** Filter natural hand tremors so the robot only moves when the user intentionally holds a stable, deliberate gesture.

* **The Human Hand Tremor Problem:**
  * Human hands are never 100% rigid; micro-tremors occur naturally every millisecond.
  * The camera streams at 30 FPS. If the robot reacted to every fractional-second fluctuation, the motors would shudder violently and trip emergency safety stops.
* **The "Hold for 1 Second" Rule (Dwell Time):**
  * A gesture must be held steadily for at least 1.0 full second before triggering actuation.
  * If a finger merely passes through a pose for 0.3 seconds during a transition, the system safely ignores it.
* **Single-Shot Latch-and-Hold Rule (Idempotency):**
  * Once the robot reaches Quadrant 1, continuing to show 1 finger does not repeatedly re-trigger motion to Quadrant 1. The robot remains stationary until a new, distinct gesture is detected.

> **Dwell Debouncer Pipeline:**  
> `Hand in Camera (With Tremor)` ➔ `Dwell Test (Held Steady ≥ 1.0s?)` ➔ `Dispatch 1 Motion Command` ➔ `Latch Lock (Prevent Repeated Triggers)`

> **Class Discussion Question:** *Have you ever observed building elevators? If you repeatedly press the elevator call button, does it arrive any faster? How does that compare to the idempotency principle used here?*

In [ ]:
#@title Initialize Dwell Debouncer Stability Filter { display-mode: "form" }
# Please do not modify the contents of this cell
import time
class TeleopDebouncer:
    """
    Gesture stability filter:
      - Dwell Time: Hand must hold gesture steadily for at least 1.0 second.
      - Latch-and-Hold: A single gesture triggers exactly 1 motion execution.
      - Cooldown: 1.0-second delay after motion completes to prevent re-triggering.
    """
    def __init__(self, dwell_time_s: float = 1.0, cooldown_s: float = 1.0):
        self.dwell_time = dwell_time_s
        self.cooldown_time = cooldown_s
        self.current_candidate = None
        self.candidate_start_time = 0.0
        self.latched_gesture = None
        self.last_trigger_time = 0.0
    def update(self, detected_gesture: str) -> bool:
        now = time.time()
        if not detected_gesture or detected_gesture in ("FIST", "UNKNOWN", "NONE"):
            self.current_candidate = None
            self.latched_gesture = None
            return False
        if detected_gesture == self.latched_gesture:
            return False
        if (now - self.last_trigger_time) < self.cooldown_time:
            return False
        if detected_gesture != self.current_candidate:
            self.current_candidate = detected_gesture
            self.candidate_start_time = now
            return False
        elapsed = now - self.candidate_start_time
        if elapsed >= self.dwell_time:
            self.last_trigger_time = now
            self.latched_gesture = detected_gesture
            self.current_candidate = None
            return True
        return False
    def get_progress(self) -> float:
        if not self.current_candidate or self.candidate_start_time == 0.0:
            return 0.0
        return min(1.0, (time.time() - self.candidate_start_time) / self.dwell_time)
    def arm_cooldown(self):
        self.last_trigger_time = time.time()
debouncer = TeleopDebouncer(dwell_time_s=1.0, cooldown_s=1.0)
print("Dwell Debouncer filter ready.")


## 3.6 Deterministic Scenario Testing (Pick and Place)

> **Quick Goal:** Execute a full end-to-end sequence (gesture mapping → lease acquisition → arm trajectory → lease release) deterministically before enabling live interactive cameras.

* **End-to-End Test Without Live Cameras:**
  * We run an automated simulation sequence: `POINT` (Quadrant 1) ➔ `OK_SIGN` (Pick) ➔ `OPEN_PALM` (Release to Box) ➔ `THUMBS_UP` (Return Home).
  * Demonstrates that waypoint interpolation, gripper actuation, and mutex leasing execute seamlessly from start to finish.
* **Safe Clearance Trajectory (Pick, Place, and Lift-Up Clearance):**
  * Observe the physical robot in the lab: quadrant transits and bin movements use smooth Joint Space (`move_j`) paths, eliminating kinematic singularities.
  * During container placement (`BOX`), the robot follows a safe 4-phase sequence: transit hover (`HOVER_BIN`), linear descent (`RELEASE_BIN`), open gripper, and **immediately lift up to safe clearance height (`HOVER_BIN`)**. This vertical clearance ensures the gripper never collides with container rims during return transits to `READY`.

> **Class Discussion Question:** *Why must the robot arm vertically lift up past the container rim before translating laterally toward the READY position? What happens if it moves sideways while still inside the box?*

In [ ]:
#@title Execute Deterministic Pick and Place Scenario { display-mode: "form" }
async def handle_gesture_trigger(gesture_name: str):
    """Accepts verified gesture name, maps action, and dispatches to robot."""
    action = GESTURE_TO_ACTION.get(gesture_name)
    if not action:
        return False
    print(f"\n[GESTURE] Detected: [{gesture_name}] -> Triggering Action: [{action}]")
    if not session.token:
        has_lock = await session.acquire_lock(duration_s=45.0)
        if not has_lock:
            return False
    # Send action along with active QUADRANT_CONFIG to lab server
    success = await session.execute(action, quadrant_config=QUADRANT_CONFIG)
    debouncer.arm_cooldown()
    return success
print("Running full deterministic Pick and Place scenario...")
scenario_gestures = ["POINT", "OK_SIGN", "OPEN_PALM", "THUMBS_UP"]
for g in scenario_gestures:
    await handle_gesture_trigger(g)
    await asyncio.sleep(1.0)
await session.release_lock()
print("Scenario execution complete.")


## 3.7 Interactive Real-Time Camera Teleoperation (Hands-on Challenge)

> **Quick Goal:** Integrate everything learned into a unified closed loop: the camera tracks your hand, AI classifies the gesture, and the physical UR5 robot arm moves in real time!

* **The Workshop Climax:**
  * All subsystems now operate synchronously in a closed loop:
    1. Your laptop camera captures hand frames locally.
    2. MediaPipe extracts 21 finger joint landmarks.
    3. The Random Forest model predicts the gesture label.
    4. The Dwell Filter verifies a 1.0-second intentional hold.
    5. The verified command streams across the internet tunnel to actuate the UR5 arm in the lab!
* **Teleoperation Guidelines:**
  * Sit or stand upright with clear lighting on your hand.
  * Present distinct gestures directly to your webcam.
  * Watch the on-screen progress bar: once it fills completely (1.0s hold), the motion command dispatches immediately to the physical robot!

> **Full Teleoperation Loop:**  
> `Laptop Webcam` ➔ `21 MediaPipe Landmarks` ➔ `Random Forest AI` ➔ `1.0s Dwell Filter` ➔ `Cloudflare Tunnel` ➔ `UR5 Physical Arm Motion`

> **Class Discussion Question:** *When switching gestures from Quadrant 1 to Quadrant 2, watch what happens to the on-screen countdown bar in your browser!*

In [ ]:
#@title Real-Time Camera Teleoperation Interface { display-mode: "form" }
# Please do not modify the contents of this cell
# 3.7 Live Webcam Teleoperation Loop in Google Colab
import base64
import time
import asyncio
import cv2
import numpy as np
from IPython.display import HTML, Javascript, display
from google.colab.output import eval_js
# JavaScript Bridge for Streaming Browser Webcam to Google Colab
LIVE_WEBCAM_JS = """
async function initTeleopWebcam() {
  const existing = document.getElementById('teleop-webcam-box');
  if (existing) existing.remove();
  const container = document.createElement('div');
  container.id = 'teleop-webcam-box';
  container.style.fontFamily = 'monospace';
  container.style.background = '#1a1a1a';
  container.style.color = '#00ffcc';
  container.style.padding = '14px';
  container.style.borderRadius = '10px';
  container.style.display = 'inline-block';
  container.style.boxShadow = '0 4px 12px rgba(0,0,0,0.5)';
  const header = document.createElement('div');
  header.innerHTML = '<span style="font-size:16px; font-weight:bold;">UR5 Gesture Teleoperation Stream</span>';
  header.style.marginBottom = '10px';
  container.appendChild(header);
  const video = document.createElement('video');
  video.id = 'teleop-video';
  video.style.borderRadius = '8px';
  video.style.transform = 'scaleX(-1)';
  video.width = 480;
  video.height = 360;
  video.autoplay = true;
  video.muted = true;
  video.playsInline = true;
  container.appendChild(video);
  const hud = document.createElement('div');
  hud.id = 'teleop-hud';
  hud.style.marginTop = '8px';
  hud.style.fontSize = '13px';
  hud.innerHTML = 'Status: Requesting webcam access...';
  container.appendChild(hud);
  const btnRow = document.createElement('div');
  btnRow.style.marginTop = '10px';
  const stopBtn = document.createElement('button');
  stopBtn.id = 'teleop-stop-btn';
  stopBtn.innerText = 'Stop Teleoperation Stream';
  stopBtn.style.background = '#e74c3c';
  stopBtn.style.color = '#fff';
  stopBtn.style.border = 'none';
  stopBtn.style.padding = '8px 16px';
  stopBtn.style.borderRadius = '5px';
  stopBtn.style.cursor = 'pointer';
  stopBtn.style.fontWeight = 'bold';
  stopBtn.onclick = () => {
    window.teleopStopped = true;
    window.teleopActive = false;
    if (window.teleopStream) {
      window.teleopStream.getTracks().forEach(track => track.stop());
    }
    stopBtn.innerText = '⏹️ Teleoperation Ended';
    stopBtn.disabled = true;
    stopBtn.style.background = '#7f8c8d';
    hud.innerHTML = 'Status: Session stopped.';
  };
  btnRow.appendChild(stopBtn);
  container.appendChild(btnRow);
  document.querySelector('#output-body').appendChild(container);
  window.teleopReady = false;
  window.teleopStopped = false;
  window.teleopActive = false;
  try {
    const stream = await navigator.mediaDevices.getUserMedia({video: {width: 480, height: 360}});
    window.teleopStream = stream;
    video.srcObject = stream;
    await video.play();
    const canvas = document.createElement('canvas');
    canvas.width = 480;
    canvas.height = 360;
    const ctx = canvas.getContext('2d');
    window.teleopActive = true;
    window.teleopReady = true;
    window.captureTeleopFrame = function() {
      if (!window.teleopActive || window.teleopStopped) return null;
      ctx.drawImage(video, 0, 0, 480, 360);
      return canvas.toDataURL('image/jpeg', 0.65);
    };
    window.updateHUD = function(gesture, conf, dwell, action) {
      const el = document.getElementById('teleop-hud');
      if (!el || window.teleopStopped) return;
      const pct = Math.round(conf * 100);
      const dwellPct = Math.round(dwell * 100);
      el.innerHTML = `Gesture: <b>${gesture}</b> (${pct}%) | Dwell: <b>${dwellPct}%</b> | Next Action: <b>${action || '-'}</b>`;
    };
    hud.innerHTML = 'Status: <span style="color:#2ecc71;">Camera Active & Ready!</span>';
  } catch (err) {
    hud.innerHTML = `<span style="color:#e74c3c;">Camera access denied/failed: ${err.message}</span>`;
    window.teleopReady = false;
    window.teleopStopped = true;
  }
}
initTeleopWebcam();
"""
def decode_base64_frame(b64_data: str) -> np.ndarray:
    if not b64_data:
        return None
    header, encoded = b64_data.split(',', 1) if ',' in b64_data else ('', b64_data)
    img_bytes = base64.b64decode(encoded)
    img_np = np.frombuffer(img_bytes, dtype=np.uint8)
    frame = cv2.imdecode(img_np, flags=cv2.IMREAD_COLOR)
    return cv2.flip(frame, 1) if frame is not None else None
async def start_live_gesture_teleop(duration_seconds: float = 180.0, conf_threshold: float = 0.50):
    """
    Real-time teleoperation loop:
      Webcam -> MediaPipe -> ML Model -> Debounce Dwell Filter -> Physical UR5 Execution
    """
    # 1. Ensure exclusive robot control lease is acquired for teleoperation session
    if not session.token:
        await session.acquire_lock(duration_s=duration_seconds + 15.0)
    # 2. Initialize camera widget in browser
    display(Javascript(LIVE_WEBCAM_JS))
    print("Waiting for camera permission from browser (click 'Allow' if prompted)...")
    camera_ready = False
    for _ in range(30):
        ready = eval_js('window.teleopReady ? true : false')
        stopped = eval_js('window.teleopStopped ? true : false')
        if ready:
            camera_ready = True
            break
        if stopped:
            break
        await asyncio.sleep(0.5)
    if not camera_ready:
        print("Camera could not be accessed or permission was denied by browser.")
        print("Tip: Click the lock icon in the Colab browser URL bar -> allow Camera (Camera: Allow), then re-run this cell.")
        if session.token:
            await session.release_lock()
        return
    print("Live Teleoperation Active! Point your hand gestures toward the camera.")
    print("Hold gesture steadily for 1.0 second to trigger UR5 arm movement.")
    print("Click the [Stop Teleoperation Stream] button on camera widget to exit.\\n")
    start_time = time.time()
    try:
        while (time.time() - start_time) < duration_seconds:
            # Check if user clicked Stop button
            user_stopped = eval_js('window.teleopStopped ? true : false')
            if user_stopped:
                print("⏹️ Teleoperation session terminated by user.")
                break
            # A. Capture base64 JPEG frame from browser
            frame_b64 = eval_js('window.captureTeleopFrame ? window.captureTeleopFrame() : null')
            if frame_b64 is None:
                await asyncio.sleep(0.05)
                continue
            # B. Decode image frame
            frame = decode_base64_frame(frame_b64)
            if frame is None:
                await asyncio.sleep(0.05)
                continue
            # C. Detect MediaPipe landmarks
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands_detector.process(rgb)
            current_gesture = "NONE"
            current_conf = 0.0
            dwell_progress = 0.0
            mapped_action = None
            if results.multi_hand_landmarks:
                hand_lms = results.multi_hand_landmarks[0]
                feats = extract_normalized_features(hand_lms)
                if feats is not None:
                    # D. Predict gesture via Random Forest
                    current_gesture, current_conf = predict_gesture_with_confidence(feats)
                    mapped_action = GESTURE_TO_ACTION.get(current_gesture)
                    # E. Dwell stability filter
                    if current_conf >= conf_threshold:
                        is_triggered = debouncer.update(current_gesture)
                        dwell_progress = debouncer.get_progress()
                        if is_triggered and mapped_action:
                            print(f"\\nVERIFIED GESTURE TRIGGER: [{current_gesture}] ({current_conf*100:.1f}%) -> Executing [{mapped_action}]!")
                            # Execute action on physical robot
                            await handle_gesture_trigger(current_gesture)
            # F. Update visual HUD in browser
            safe_act = mapped_action if mapped_action else "-"
            eval_js(f'window.updateHUD ? window.updateHUD("{current_gesture}", {current_conf}, {dwell_progress}, "{safe_act}") : null')
            await asyncio.sleep(0.04)  # ~25 FPS pacing loop
    except asyncio.CancelledError:
        print("Teleoperation task cancelled.")
    except Exception as e:
        print(f"Exception in teleop loop: {e}")
    finally:
        # Cleanup browser camera track
        eval_js('if (window.teleopStream) { window.teleopStream.getTracks().forEach(t => t.stop()); window.teleopActive = false; window.teleopStopped = true; }')
        if session.token:
            await session.release_lock()
        print("Teleoperation session finished. Robot lock safely released.")
# Run live camera teleoperation (default active duration: 180 seconds / 3 minutes)
await start_live_gesture_teleop(duration_seconds=180.0, conf_threshold=0.50)


## 3.8 Natural Language Commands to the UR5 with an LLM

> **Quick Goal:** Replace the gesture trigger from Section 3.7 with a single sentence. The camera and Random Forest are still used for earlier practice; now the LLM selects an action from the server's vocabulary.

### What Changes from 3.7?

| Aspect | 3.7 Gesture Teleoperation | 3.8 LLM Command |
| --- | --- | --- |
| Trigger | Random Forest label | Natural-language sentence |
| Action selector | `GESTURE_TO_ACTION` | Reactive LLM in `llm_engine.py` (called by `robot_command_server.py`) |
| Colab → lab link | `RobotSession` (`session`) | The **same** `RobotSession` (`session`) |
| Lease | `acquire_lock()` / `release_lock()` | Identical |
| Guardrails | Server | Server |

### Cells in This Section

* **3.8.1** — initialize the item list on the server (`SET_ITEMS`).
* **3.8.2** — prompt, model, and planning mode. Choose `smart` or `flash`; the lab server resolves that label to the actual model and API key. `reactive` decides one action at a time (default); `full plan` asks the LLM for the whole action list once, dry-runs it against the guardrails, executes it, then asks whether the task is finished (replanning if needed). Jev is rejected in full-plan mode. The cell also shows the system prompt and the live item positions.
* **3.8.3** — send `PROMPT_TASK`, show the response, and refresh the robot position after every action.

### Message Flow

`Prompt` ➔ `session.ws_url` ➔ `PROMPT_TASK` ➔ `PLAN_ACK` ➔ `STEP_PROGRESS` ➔ `PLAN_COMPLETE`

### What Is Not Repeated

This cell does **not** redefine `RobotSession`, `_msg()`, the tunnel URL, or the lease mechanism. It reuses `session` and `_msg()` from Chapter 3.1 directly. The WebSocket opens at `session.ws_url` and the payload is built with `_msg()` — exactly like `session.execute()` for gestures.

### System Prompt — Read-Only, Server-Side

The planner's system prompt is **embedded in the lab server**, not in this notebook. The authoritative copy is the `REACTIVE_SYSTEM_PROMPT` constant in `llm_engine.py`, used by `robot_main_server.py` through `ReactiveLLMMixin`. Cell 3.8.2 shows a read-only copy written directly in that cell — it is not fetched over the wire and nothing here can change the server's rules. The `CURRENT ITEM POSITIONS (from robot server)` line is refreshed from `robot_feedback` on every `STEP_PROGRESS`, so it tracks the live item-to-square map while the arm works.

It fixes the rules the LLM must follow: the 64-square chessboard workspace, only `pick` / `place` / `move` / `move to home` (plus `task_completed` / `reject_task`), live item positions as the only ground truth, **one action per turn**, action history, and the guardrails the server enforces. `move` travels above the chosen square at the same height as the end of a pick/place — it does not touch the gripper or any item position.

In Section 2.4 you were free to rewrite the system instruction. Here the equivalent rule set is locked on the server so a client cannot weaken a hardware safety rule. To add a preference, put it in the **task prompt** of cell 3.8.2 as extra notes — the server rules still take precedence.

> As in the chat of Section 2.2, Colab **does not send an API key**. The real key stays on the lab PC and is used by the server when calling the LLM.

> There is no E-STOP button in this notebook. `EMERGENCY_STOP` can still be sent by any client (e.g. `commander_station.py`); the server then cancels the active task and, in hardware mode, sends an RTDE stop so the current motion halts.

> A task such as *"Pick item 1 and place it on pos 1"* can finish in **zero steps** if the item is already at the destination. That follows the target-position rule, not a malfunction.

> **Class Discussion Question:** *Why must the server validate the LLM's action even though the LLM already knows the rule list?*


In [ ]:
#@title 3.8.1 Initialize Item Positions & Item List { display-mode: "form" }
# First cell: initialize item positions and the item list on the robot command server.

import asyncio
import html as _html
import json
import threading
import ipywidgets as widgets
import websockets
from IPython.display import display

if "session" not in globals() or "_msg" not in globals():
    raise RuntimeError("Run Chapter 3.1 first (RobotSession / _msg) before cell 3.8.1.")

SQUARES = [f"{c}{r}" for c in "abcdefgh" for r in range(1, 9)]
server_item_snapshot = {"status": "-", "current": {}, "initial": {}, "holding": None, "robot_feedback": {}}

items_panel = widgets.HTML(value="")
new_name = widgets.Text(
    value="", placeholder="item name, e.g. item_4", description="New item:",
    layout=widgets.Layout(width="230px"),
)
new_pos = widgets.Dropdown(
    options=SQUARES, value="a1", description="Square:",
    layout=widgets.Layout(width="180px"),
)
add_btn = widgets.Button(description="Add Item", button_style="primary", icon="plus")
reload_btn = widgets.Button(description="Reload", button_style="info", icon="refresh")
clear_btn = widgets.Button(description="Clear All", button_style="danger", icon="trash")
rows_box = widgets.VBox([])
apply_btn = widgets.Button(description="Apply to Server", button_style="success", icon="check")
status_html = widgets.HTML(value="")


def _set_status(text, color=None):
    status_html.value = (
        f"<span style='color:{color or '#868e96'};font-size:13px;'>{_html.escape(str(text))}</span>"
    )


def render_items_panel():
    cur = server_item_snapshot.get("current", {}) or {}
    ini = server_item_snapshot.get("initial", {}) or {}
    rows = []
    for name in sorted(set(cur) | set(ini)):
        rows.append(
            "<tr>"
            f"<td style='padding:3px 8px;'>{_html.escape(str(name))}</td>"
            f"<td style='padding:3px 8px;color:#a6e3a1;'>{_html.escape(str(cur.get(name, '-')))}</td>"
            f"<td style='padding:3px 8px;color:#89b4fa;'>{_html.escape(str(ini.get(name, '-')))}</td>"
            "</tr>"
        )
    if not rows:
        rows.append("<tr><td colspan='3' style='padding:6px;color:#6c7086;'>No items yet.</td></tr>")
    items_panel.value = (
        "<div style='border:1px solid #454a55;border-radius:8px;padding:10px;background:#11151c;'>"
        "<b style='color:#89b4fa;'>ITEM STATUS / ROBOT COMMAND SERVER</b>"
        "<div style='color:#6c7086;font-size:12px;'>"
        f"status: {_html.escape(str(server_item_snapshot.get('status')))}"
        f" | gripper: {_html.escape(str(server_item_snapshot.get('holding')))}"
        "</div>"
        "<table style='border-collapse:collapse;font-family:monospace;font-size:12px;"
        "color:#cdd6f4;margin-top:6px;'>"
        "<tr style='color:#f9e2af;'>"
        "<th style='padding:3px 8px;text-align:left;'>Item</th>"
        "<th style='padding:3px 8px;text-align:left;'>Current</th>"
        "<th style='padding:3px 8px;text-align:left;'>Initial</th></tr>"
        + "".join(rows) + "</table></div>"
    )


async def refresh_server_items():
    async with websockets.connect(session.ws_url, open_timeout=10) as ws:
        await ws.send(_msg("HANDSHAKE", {"client_id": session.student_id, "version": "2.0"}))
        hs = json.loads(await asyncio.wait_for(ws.recv(), timeout=10))
    p = hs.get("payload", {})
    fb = p.get("robot_feedback", {}) or {}
    server_item_snapshot.update({
        "status": p.get("server_status", "-"),
        "current": dict(p.get("current_item_positions", {})),
        "initial": dict(p.get("initial_item_positions", {})),
        "holding": fb.get("holding_item"),
        "robot_feedback": fb,
    })
    render_items_panel()
    if "_render_current_info" in globals():
        _render_current_info()
    return server_item_snapshot


def make_row(name="", square="a1"):
    square = str(square)
    if square not in SQUARES:
        square = SQUARES[0]
    name_w = widgets.Text(value=str(name), description="Item:", layout=widgets.Layout(width="230px"))
    pos_w = widgets.Dropdown(
        options=SQUARES, value=square, description="Square:",
        layout=widgets.Layout(width="180px"),
    )
    del_w = widgets.Button(
        description="Remove", button_style="danger", icon="trash",
        layout=widgets.Layout(width="95px"),
    )
    row = widgets.HBox([name_w, pos_w, del_w])

    def _del(_):
        if row in rows_box.children:
            rows_box.children = tuple(r for r in rows_box.children if r is not row)

    del_w.on_click(_del)
    return row


def populate_rows_from_server():
    cur = server_item_snapshot.get("current", {}) or {}
    rows_box.children = tuple(make_row(n, cur[n]) for n in sorted(cur))


def build_items():
    items, seen = {}, {}
    for row in rows_box.children:
        name = row.children[0].value.strip()
        pos = row.children[1].value
        if not name:
            continue
        if name in items:
            raise ValueError(f"Duplicate item: {name}")
        if pos in seen:
            raise ValueError(f"Petak {pos} dipakai {seen[pos]} dan {name}")
        items[name] = pos
        seen[pos] = name
    return items


def _on_add(_):
    name = new_name.value.strip()
    if not name:
        _set_status("Item name is empty.", "#f9e2af")
        return
    pos = new_pos.value
    names = {r.children[0].value.strip() for r in rows_box.children}
    used = {r.children[1].value for r in rows_box.children}
    if name in names:
        _set_status(f"Item '{name}' already exists.", "#f38ba8")
        return
    if pos in used:
        _set_status(f"Square {pos} is already used.", "#f38ba8")
        return
    rows_box.children = rows_box.children + (make_row(name, pos),)
    new_name.value = ""
    _set_status(f"Item '{name}' added at {pos}.", "#a6e3a1")


def _on_clear(_):
    rows_box.children = ()
    _set_status("All rows removed. Click Apply to Server to save.", "#f9e2af")


def _on_reload(_):
    async def _reload():
        await refresh_server_items()
        populate_rows_from_server()
        _set_status("Rows reloaded from server. Edit name/square or click Remove on a row.", "#a6e3a1")
    _fire(_reload())


def _fire(coro):
    def _runner():
        try:
            asyncio.run(coro)
        except Exception as exc:
            _set_status(f"Failed: {exc}", "#f38ba8")
    t = threading.Thread(target=_runner, daemon=True)
    t.start()
    return t


async def _apply_items(items):
    _set_status("Sending SET_ITEMS to server...", "#f9e2af")
    async with websockets.connect(session.ws_url, open_timeout=10) as ws:
        await ws.send(_msg("HANDSHAKE", {"client_id": session.student_id, "version": "2.0"}))
        await asyncio.wait_for(ws.recv(), timeout=10)
        await ws.send(_msg("SET_ITEMS", {
            "items": items, "mode": "initial", "client_id": session.student_id,
        }))
        while True:
            m = json.loads(await asyncio.wait_for(ws.recv(), timeout=10))
            if m.get("msg_type") == "SET_ITEMS_ACK":
                p = m.get("payload", {})
                _set_status(
                    ("OK | " if p.get("success") else "REJECTED | ") + str(p.get("message")),
                    "#a6e3a1" if p.get("success") else "#f38ba8",
                )
                break
            if m.get("msg_type") == "ERROR":
                _set_status("ERROR | " + str(m.get("payload", {}).get("error")), "#f38ba8")
                break
    await refresh_server_items()
    populate_rows_from_server()


def _on_apply(_):
    try:
        items = build_items()
    except Exception as exc:
        _set_status(str(exc), "#f38ba8")
        return
    _fire(_apply_items(items))


add_btn.on_click(_on_add)
reload_btn.on_click(_on_reload)
clear_btn.on_click(_on_clear)
apply_btn.on_click(_on_apply)
try:
    await refresh_server_items()
    populate_rows_from_server()
except Exception as exc:
    _set_status(f"Failed to read server status: {exc}", "#f38ba8")

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;color:#89b4fa;'>SET ITEMS & SQUARES</h3>"),
    items_panel,
    widgets.HBox([new_name, new_pos, add_btn], layout=widgets.Layout(margin="6px 0")),
    widgets.HTML(
        "<div style='color:#6c7086;font-size:12px;'>"
        "Edit name/square directly in the rows below. Click Remove on a row to delete an item."
        "</div>"
    ),
    rows_box,
    widgets.HBox([reload_btn, clear_btn, apply_btn], layout=widgets.Layout(margin="6px 0")),
    status_html,
]))
print("Item initialization ready.")


In [ ]:
#@title 3.8.2 Natural Language Command → UR5 (Smart / Flash) { display-mode: "form" }
# Prompt and model input. Item positions are read automatically from the robot command server.

import asyncio
import html as _html
import ipywidgets as widgets
from IPython.display import display

if "session" not in globals() or "_msg" not in globals():
    raise RuntimeError("Run Chapter 3.1 first (RobotSession / _msg).")
if "refresh_server_items" not in globals():
    raise RuntimeError("Run cell 3.8.1 first (item initialization).")

MODEL_OPTIONS = [
    ("smart", "smart"),
    ("flash", "flash"),
]


prompt_box = widgets.Text(
    value="Pick up the item at a1 and place it at b4",
    description="Task:", layout=widgets.Layout(width="520px"),
)
model_dropdown = widgets.Dropdown(
    options=MODEL_OPTIONS, value="smart", description="Model:",
)
plan_mode_dropdown = widgets.Dropdown(
    options=[("reactive", "reactive"), ("full plan", "full")],
    value="reactive", description="Planning:",
)
current_info = widgets.HTML(value="")

SYSTEM_PROMPT_TEXT = """You are an intelligent closed-loop action planner for a UR5 arm with a Robotiq 2F-85 gripper on an 8x8 chessboard.
You receive the COMPLETE user task and choose EXACTLY ONE action per turn.
There is NO fixed number of pick/place cycles: execute EVERY instruction, in order.

WORKSPACE MODEL:
- The 64 squares a1..h8 are calibrated poses from captured_points.json. No IK, no step motions.
- CURRENT ITEM POSITIONS maps each item to its square (or 'gripper'). This is the ONLY ground truth.
- HOLDING ITEM tells which item is in the gripper; at most one item can be held.
- Each square holds at most one item. Do NOT place onto an occupied square.

ALLOWED ACTIONS (4 only):
- 'pick': take an item from a square. 'square' is the SOURCE and is required.
- 'place': release the held item. 'square' is the DESTINATION and is required.
- 'move': travel to the square's approach height (same height as after pick/place). 'square' is the DESTINATION and is required.
- 'move to home': return the arm to calibrated HOME_POS. Set 'square' to 'none'.
- 'task_completed' / 'reject_task' as described below.

HOW TO INTERPRET THE TASK:
- The tool has NO item parameter. If the user says 'item_1' / 'barang 1', look it up in CURRENT ITEM POSITIONS and emit 'pick' with THAT SQUARE.
- 'ambil / pick / grasp / angkat dari <square>' = PICK ONLY: emit 'pick' with square='<square>'. KEEP holding; do NOT place it.
- 'taruh / place / letakkan ke <square>' = PLACE ONLY: emit 'place' with square='<square>'. The item may already be in the gripper.
- 'move / pindah / hover ke <square>' = MOVE ONLY: emit 'move' with square='<square>'. Travel at the elevated approach height; do NOT open/close the gripper and do NOT change any item position. Holding an item is allowed - it stays in the gripper.
- MULTI-STEP prompts: repeat for EACH instruction in order. Do NOT call 'task_completed' after the first one if more remain.
- A combined 'pick X then place on Y' needs two turns: 'pick' first, 'place' on a later turn.

PROGRESS TRACKING — READ ACTION HISTORY CAREFULLY:
- If 'pick' of that item and 'place' at that square already appear in ACTION HISTORY, that cycle is DONE. Move on; never repeat it.
- If the item is already at the destination, that instruction is satisfied — skip it.
- After 'pick', HOLDING ITEM is set; after 'place' it clears and the item's position becomes the destination square.

GROUND TRUTH & GUARDRAILS:
- Never pick while HOLDING ITEM is set; never place when it is empty.
- Never pick from an empty square and never place onto an occupied square. The server rejects both and feeds the error back for self-correction.
- Trust BOARD OCCUPANCY and HOLDING ITEM over assumptions.

COMPLETION & REJECTION:
- Call 'task_completed' ONLY when EVERY instruction is done. A pick-only task correctly ends with the item still held.
- Call 'reject_task' only at the very start if the task is impossible. After any successful pick/place, NEVER reject — finish the remaining instructions.

ACTION HISTORY:
- Turn 1: ACTION HISTORY is None. Later turns: full list of executed actions. Never repeat completed steps.

Call select_next_arm_action with action, square, is_completed and reasoning."""

system_prompt_panel = widgets.HTML(value=(
    "<div style='border:1px solid #454a55;border-radius:8px;padding:10px;"
    "background:#11151c;max-height:240px;overflow:auto;'>"
    "<b style='color:#cba6f7;'>SYSTEM PROMPT (SERVER-SIDE, READ-ONLY)</b>"
    "<pre style='white-space:pre-wrap;color:#cdd6f4;font-family:monospace;"
    "font-size:11px;margin:6px 0 0 0;'>" + _html.escape(SYSTEM_PROMPT_TEXT) + "</pre></div>"
))


def _render_current_info():
    current_info.value = (
        "<div style='color:#6c7086;font-size:12px;margin:6px 0;'>"
        f"CURRENT ITEM POSITIONS (from robot server): {_html.escape(str(server_item_snapshot.get('current')))}"
        "</div>"
    )


def _apply_robot_feedback(feedback):
    """Refresh posisi item dari robot_feedback tiap aksi."""
    fb = feedback or {}
    server_item_snapshot["holding"] = fb.get("holding_item")
    server_item_snapshot["robot_feedback"] = fb
    if fb.get("current_item_positions"):
        server_item_snapshot["current"] = dict(fb["current_item_positions"])
    if fb.get("initial_item_positions"):
        server_item_snapshot["initial"] = dict(fb["initial_item_positions"])
    _render_current_info()
    if "render_items_panel" in globals():
        render_items_panel()


try:
    await refresh_server_items()
    _render_current_info()
except Exception as exc:
    current_info.value = f"<div style='color:#f38ba8;font-size:12px;'>Failed to read positions: {_html.escape(str(exc))}</div>"

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;color:#89b4fa;'>NATURAL LANGUAGE / UR5</h3>"),
    widgets.HBox([model_dropdown, plan_mode_dropdown, prompt_box], layout=widgets.Layout(margin="6px 0")),
    current_info,
    widgets.HTML("<div style='color:#6c7086;font-size:12px;margin:8px 0 4px 0;'>"
                 "System prompt below is sent by the lab server. It is read-only.</div>"),
    system_prompt_panel,
]))
print("Form ready. Fill the prompt, choose a model, then run cell 3.8.3.")


In [ ]:
#@title 3.8.3 Send to Robot & Display Response { display-mode: "form" }
# Run this cell AFTER filling the prompt and choosing a model in cell 3.8.2.

import asyncio
import html as _html
import json
import ipywidgets as widgets
import websockets
from IPython.display import display

if "session" not in globals() or "_msg" not in globals():
    raise RuntimeError("Run Chapter 3.1 first (RobotSession / _msg).")
if "prompt_box" not in globals() or "model_dropdown" not in globals():
    raise RuntimeError("Run cell 3.8.2 before cell 3.8.3.")

prompt = prompt_box.value.strip()
model = model_dropdown.value
plan_mode = plan_mode_dropdown.value if "plan_mode_dropdown" in globals() else "reactive"

panel = widgets.HTML(value="")
_lines = []
display(panel)
reason_panel = widgets.HTML(value="")
display(reason_panel)
_reason = {"text": "", "step": None}


def _render_reason():
    safe = _html.escape(_reason["text"]).replace("\n", "<br>")
    body = safe or "<span style='color:#6c7086;'>Waiting for reasoning...</span>"
    reason_panel.value = (
        "<div style='border:1px solid #454a55;border-radius:8px;padding:8px;"
        "background:#11151c;color:#cba6f7;font-family:monospace;font-size:12px;line-height:1.5;'>"
        "<b>REASONING (per step)</b><br>" + body + "</div>"
    )


_render_reason()


def _line(text: str, color: str = "#cdd6f4", bold: bool = False):
    safe = _html.escape(str(text)).replace("\n", "<br>")
    weight = "font-weight:bold;" if bold else ""
    _lines.append(
        f"<div style='border-left:3px solid {color};padding:3px 8px;margin:3px 0;"
        f"font-family:monospace;font-size:12px;line-height:1.5;{weight}color:{color};'>{safe}</div>"
    )
    panel.value = (
        "<div style='border:1px solid #454a55;border-radius:8px;padding:10px;"
        "background:#11151c;max-height:360px;overflow:auto;'>"
        + "".join(_lines) + "</div>"
    )


_line(f"PROMPT: {prompt}", "#89b4fa", True)
_line(f"MODEL: {model}", "#89b4fa", True)

if not prompt:
    _line("Prompt is empty. Fill it in cell 3.8.2.", "#f38ba8")
else:
    acquired = False
    try:
        if not session.token:
            acquired = await session.acquire_lock(duration_s=120.0)
        async with websockets.connect(session.ws_url, open_timeout=10) as ws:
            await ws.send(_msg("HANDSHAKE", {"client_id": session.student_id, "version": "2.0"}))
            hs = json.loads(await asyncio.wait_for(ws.recv(), timeout=10))
            hp = hs.get("payload", {})
            _line(f"SERVER: {hp.get('server_status')} | item: {hp.get('current_item_positions')}", "#6c7086")
            _apply_robot_feedback(hp.get("robot_feedback") or {})
            payload = {"prompt": prompt, "model": model, "plan_mode": plan_mode}
            if session.token:
                payload["token"] = session.token
            await ws.send(_msg("PROMPT_TASK", payload))
            _line(f"SENT PROMPT_TASK ({model} | {plan_mode})", "#f9e2af")
            first = True
            while True:
                try:
                    raw = await asyncio.wait_for(ws.recv(), timeout=15 if first else 180)
                except asyncio.TimeoutError:
                    _line("TIMEOUT: waiting for server.", "#f38ba8")
                    break
                first = False
                msg = json.loads(raw)
                mt = msg.get("msg_type")
                p = msg.get("payload", {})
                if p.get("robot_feedback"):
                    _apply_robot_feedback(p["robot_feedback"])
                if mt == "PLAN_ACK":
                    _line(f"ACK: {p.get('message', p.get('status', ''))}", "#89b4fa")
                    if p.get("accepted") is False:
                        break
                elif mt == "PLAN_REJECTED":
                    head = (f"cycle {p.get('cycle')}" if p.get("cycle")
                            else f"step {p.get('step_index')}")
                    plan_txt = ""
                    if p.get("actions"):
                        plan_txt = " | plan: " + ", ".join(
                            f"{a.get('action')} {a.get('square')}".strip()
                            for a in p["actions"]
                        )
                    _line(
                        f"PLAN REJECTED [{head}, retry {p.get('retry')}] "
                        f"{p.get('kind')}: {p.get('reason')}{plan_txt}",
                        "#fab387",
                    )
                elif mt == "STEP_PROGRESS":
                    step = p.get('step_index', '?')
                    act = p.get('action', '')
                    if p.get("status") == "SUCCESS":
                        _line(f"  SUCCESS [{step}] {act}", "#a6e3a1")
                    else:
                        _line(f"  RUNNING [{step}] {act} -> {p.get('status')}", "#f9e2af")
                elif mt == "REASONING":
                    step_no = p.get("step_index", "?")
                    if _reason["step"] != step_no:
                        _reason["step"] = step_no
                        prefix = f"[step {step_no}] " if not _reason["text"] else f"\n[step {step_no}] "
                        _reason["text"] += prefix
                    _reason["text"] += p.get("chunk", "")
                    _render_reason()
                elif mt == "PLAN_COMPLETE":
                    status = p.get("task_status") or p.get("status")
                    _line(
                        f"COMPLETE: {status} | turn {p.get('turn')} | "
                        f"{p.get('total_duration_s', 0):.1f}s",
                        "#a6e3a1" if status == "FINISH" else "#f38ba8", True,
                    )
                    break
                elif mt == "ERROR":
                    _line(f"ERROR: {p.get('error')}", "#f38ba8")
                    break
    except Exception as exc:
        _line(f"FAILED: {exc}", "#f38ba8")
    finally:
        if acquired:
            await session.release_lock()
        _line("DONE - cell stopped.", "#6c7086")

if "refresh_server_items" in globals():
    try:
        await refresh_server_items()
        _render_current_info()
    except Exception as exc:
        print(f"[INFO] Failed to refresh items: {exc}")


## 3.9 Rangkuman Praktikum Hands-on UR5

> **Quick Goal:** Consolidate the 5 core engineering takeaways mastered during today's workshop.

* **1. Distributed Robotics Architecture:** Your laptop performs computer vision processing locally, while the lab workstation handles physical motor execution.
* **2. Shared Resource Management (Mutex Lease):** Safely sharing a single physical robot arm across dozens of simultaneous users without instruction collisions.
* **3. Intelligent Geometry Over Raw Pixels:** 21 landmark skeletons make our AI exceptionally lightweight, fast, and invariant to room lighting or background clutter.
* **4. Signal Conditioning (Dwell Debouncer):** Eliminates natural human hand tremors, ensuring smooth, deterministic, and safe robotic actuation.
* **5. Clearance Trajectory Safety (Orthogonal L-Shape):** Enforces vertical elevation before lateral transit, guaranteeing the arm clears all obstacles and container rims.

# Chapter 4: Interaction and Control of Quadruped Mobile Robots (RoboDog)

In Chapter 4, we connect hand-pose perception to a four-legged quadruped robot (**DeepRobotics Lite3 RoboDog**). Unlike the UR5 arm in Chapter 3—which used a Random Forest classifier and WebSocket waypoint leases—here you will record one custom reference pose per robot command and stream real-time velocity and posture commands over an HTTP REST bridge.

This chapter is organized into **three parts**:
1. **Part 1 (Section 4.1 — Lite3 Hand Pose Registry)**: Teach the computer your custom hand shapes using your webcam and save them into `gestures.json`.
2. **Part 2 (Section 4.2 — Lite3 Hand Pose Detection & REST Publisher)**: Use those saved hand shapes in `gestures.json` to steer the robot dog live over your local Wi-Fi.
3. **Part 3 (Section 4.3 — Natural-Language LLM Control for Lite3 RoboDog)**: Engineer a system prompt and few-shot examples so an LLM translates everyday English or Indonesian instructions into safe `geometry_msgs/Twist` commands.

> **Environment Note:** If you already ran **Chapter 0** and **Chapter 1.1**, the required Python libraries (`mediapipe`, `opencv-python-headless`, `requests`, `numpy`) and the `hand_landmarker.task` model are already installed and ready—Section 4.1 will reuse them automatically.

---

## 4.1 Part 1: Lite3 Hand Pose Registry

### What Is a "Pose Class"?
In machine learning, a **class** is simply a named category—just like labeling photo folders `"cat"` and `"dog"`. Here, each class is one hand shape linked to one robot command. During setup, we record one sample pose per class into `gestures.json`:

| Class Name | What the Robot Does | Hand Shape Idea (Pick Clear, Different Poses!) |
|---|---|---|
| `forward` | Walks forward | Open flat palm or 1 finger pointing up |
| `backward` | Walks backward | Thumbs down or fist |
| `left` | Sidesteps left | Thumb pointing left / L-shape |
| `right` | Sidesteps right | Pinky/thumb pointing right |
| `turn_left` | Rotates counter-clockwise | Peace sign (2 fingers) |
| `turn_right` | Rotates clockwise | 3 fingers up |
| `sit` | Robot sits / lies down | OK sign (thumb + index circle) |
| `stand` | Robot stands up | Thumbs up |
| `stop` | Safety stop (all movement speeds set to `0`) | Closed fist or flat stop hand |

> **Beginner Tip:** Pick hand poses that look clearly different from one another (try not to use two poses that only change one pinky angle), and keep your whole hand inside the camera view.

### How Setup Works
1. **Hand Landmarker Reuse**: Reuses `hand_landmarker.task` from Chapter 0 (downloading it only if missing) and reuses the MediaPipe Tasks `HandLandmarker` initialized in Chapter 1.1 if available.
2. **Prepare Classes & Pose Math (`4.1.1`)**:
   - Sets up the **9 `POSE_CLASSES`** listed above.
   - **Normalizes landmarks (`normalize_landmarks`)**: Centers the 21 hand dots on your wrist and scales them by hand size so your pose still matches whether your hand is close to or far from the webcam.
   - **Sets `TOLERANCE` (`0.15`)**: Controls how strict the pose match is (lower = stricter match required; higher = more forgiving).
   - **Defines `GESTURES_PATH` (`gestures.json`)**: The file where your recorded poses are stored. If you already saved a `gestures.json` earlier and want to add/edit poses, upload it in Colab's left **Files** panel before running `4.1.2`.

### 4.1.1 Configure Pose Classes & Load Landmark Helpers

You do not need to edit anything here—just click **Play (`▶`)** to load the 9 `POSE_CLASSES`, default `TOLERANCE` (`0.15`), and landmark helper functions. Look for **`Setup complete. Loaded gestures: [...]`** below the cell when finished (`[]` is normal if you haven't recorded any poses yet).

In [ ]:
#@title 4.1.1 Configure Pose Classes & Load Landmark Helpers { display-mode: "form" }
import base64
import json
import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks.python import vision

try:
    BaseOptions = mp.tasks.BaseOptions
except AttributeError:
    from mediapipe.tasks.python import core
    BaseOptions = core.base_options.BaseOptions

MODEL_PATH = Path("hand_landmarker.task")
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/1/hand_landmarker.task"
)
if not MODEL_PATH.exists():
    print("Downloading hand_landmarker.task...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

GESTURES_PATH = Path("gestures.json")
TOLERANCE = 0.15
POSE_CLASSES = (
    "forward",
    "backward",
    "left",
    "right",
    "turn_left",
    "turn_right",
    "sit",
    "stand",
    "stop",
)
HAND_CONNECTIONS = (
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17),
)


def draw_hand_skeleton(frame: np.ndarray, hand_landmarks) -> None:
    if hasattr(vision, "drawing_utils") and hasattr(vision, "drawing_styles") and hasattr(vision, "HandLandmarksConnections"):
        vision.drawing_utils.draw_landmarks(
            frame,
            hand_landmarks,
            vision.HandLandmarksConnections.HAND_CONNECTIONS,
            vision.drawing_styles.get_default_hand_landmarks_style(),
            vision.drawing_styles.get_default_hand_connections_style(),
        )
        return
    h, w = frame.shape[:2]
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]
    for a, b in HAND_CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], (0, 220, 0), 2, cv2.LINE_AA)
    for x, y in pts:
        cv2.circle(frame, (x, y), 4, (0, 80, 255), -1, cv2.LINE_AA)


def normalize_landmarks(hand_landmarks) -> np.ndarray:
    coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks], dtype=np.float32)
    centered = coords - coords[0]
    scale = float(np.max(np.linalg.norm(centered, axis=1)))
    return centered / scale if scale > 1e-6 else centered


def load_gestures(path: str | Path = GESTURES_PATH) -> dict[str, np.ndarray]:
    p = Path(path)
    if not p.exists():
        return {}
    raw = json.loads(p.read_text(encoding="utf-8"))
    return {name: np.asarray(coords, dtype=np.float32) for name, coords in raw.items()}


def save_gesture(path: str | Path, name: str, coords: np.ndarray) -> dict[str, np.ndarray]:
    p = Path(path)
    gestures = load_gestures(p)
    gestures[name] = coords.astype(np.float32)
    serializable = {k: np.round(v, 6).tolist() for k, v in gestures.items()}
    p.write_text(json.dumps(serializable, indent=2) + "\n", encoding="utf-8")
    return gestures


def classify_gesture(
    live_coords: np.ndarray,
    gestures: dict[str, np.ndarray],
    tolerance: float = TOLERANCE,
) -> tuple[str, float | None, str | None]:
    if not gestures:
        return "No gestures saved", None, None
    best_name, best_dist = None, float("inf")
    for name, ref_coords in gestures.items():
        if ref_coords.shape != live_coords.shape:
            continue
        dist = float(np.mean(np.linalg.norm(live_coords - ref_coords, axis=1)))
        if dist < best_dist:
            best_dist, best_name = dist, name
    if best_name is None:
        return "No gestures saved", None, None
    if best_dist <= tolerance:
        return best_name, best_dist, best_name
    return "Unknown", best_dist, best_name


def decode_data_url(data_url: str) -> np.ndarray | None:
    if not data_url or "," not in data_url:
        return None
    encoded = data_url.split(",", 1)[1]
    if not encoded:
        return None
    buf = np.frombuffer(base64.b64decode(encoded), dtype=np.uint8)
    if buf.size == 0:
        return None
    return cv2.imdecode(buf, cv2.IMREAD_COLOR)


def encode_data_url(frame: np.ndarray) -> str:
    ok, buf = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 80])
    if not ok:
        raise RuntimeError("Failed to encode frame")
    return "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode("ascii")


def create_image_landmarker() -> vision.HandLandmarker:
    global _shared_hand_landmarker
    if "_shared_hand_landmarker" in globals() and _shared_hand_landmarker is not None:
        return _shared_hand_landmarker
    if (
        "hands_detector" in globals()
        and getattr(hands_detector, "mode", None) == "tasks"
        and getattr(hands_detector, "detector", None) is not None
    ):
        _shared_hand_landmarker = hands_detector.detector
        return _shared_hand_landmarker
    options = vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=vision.RunningMode.IMAGE,
        num_hands=1,
    )
    _shared_hand_landmarker = vision.HandLandmarker.create_from_options(options)
    return _shared_hand_landmarker


_landmarker_source = (
    "reusing Chapter 1.1 HandLandmarker"
    if (
        "hands_detector" in globals()
        and getattr(hands_detector, "mode", None) == "tasks"
        and getattr(hands_detector, "detector", None) is not None
    )
    else f"model ready at {MODEL_PATH.resolve()}"
)
print(f"Setup complete ({_landmarker_source}). Loaded gestures:", list(load_gestures().keys()))

### 4.1.2 Interactive Pose Registration (Select Class & Save Pose)

Follow these steps to record your personal hand shapes for each class:

1. **Start the camera**: Click **Play (`▶`)** on the cell below, then click **Allow** when your browser asks for webcam permission.
2. **Pick a class**: Click one of the 9 buttons above the video (for example `forward`):
   - **Blue button with yellow border**: Currently selected class ready to save.
   - **Green button with `*`**: Class already has a saved pose inside `gestures.json`.
3. **Save your pose**: Hold that hand pose clearly inside the camera frame until the skeleton lines appear on your hand, then click **SAVE POSE**. You will see `Saved 'forward' (1/9)`.
4. **Repeat & test live recognition**: Record all 9 classes (or at least the commands you plan to use). Once saved, switch hand poses and check the text at the top-left of the video (`Detected: forward (forward: 0.042)`). Smaller error numbers (`err`) mean a closer match!
5. **Finish recording**: Click the red **STOP** button to turn off your webcam before moving on to Section 4.2 (or optionally download `gestures.json` in `4.1.3`).

#### Troubleshooting
- **Camera blocked / error**: Make sure no other app (Zoom, Teams, or another Colab tab) is using your webcam, then click the camera/lock icon in Chrome's URL bar to allow camera access and re-run the cell.
- **Always says `Unknown` even for a saved pose**: Raise the **Tolerance** box slightly (e.g., from `0.15` to `0.20`), or choose hand poses that look more distinct from each other.
- **Want to replace a bad pose?** Simply click that class's button again, hold your updated hand pose, and click **SAVE POSE**—it overwrites the old sample for that class automatically.

In [ ]:
#@title 4.1.2 Interactive Hand Pose Registration Studio { display-mode: "form" }
from google.colab import output
from IPython.display import HTML, JSON, display

_reg_landmarker = create_image_landmarker()
_reg_gestures = load_gestures(GESTURES_PATH)


def _register_frame_cb(data_url: str, target_class: str, do_save: bool, tolerance: float):
    global _reg_gestures
    frame = decode_data_url(data_url)
    if frame is None:
        return JSON({"image": "", "status": "Waiting for valid frame...", "saved": list(_reg_gestures.keys())})

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = _reg_landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))

    status = "No hand detected"
    primary_coords = None

    if res.hand_landmarks:
        hand_lms = res.hand_landmarks[0]
        draw_hand_skeleton(frame, hand_lms)
        primary_coords = normalize_landmarks(hand_lms)
        label, dist, closest = classify_gesture(primary_coords, _reg_gestures, float(tolerance))
        color = (0, 220, 0) if label not in ("Unknown", "No gestures saved") else (0, 180, 255)
        text = f"{label}" + (f" ({closest}: {dist:.3f})" if dist is not None else "")
        cv2.putText(frame, text, (16, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)
        status = f"Detected: {text}"

    if do_save:
        if primary_coords is None:
            status = "Cannot save: show your hand in frame first!"
        elif target_class in POSE_CLASSES:
            _reg_gestures = save_gesture(GESTURES_PATH, target_class, primary_coords)
            status = f"Saved '{target_class}' ({len(_reg_gestures)}/{len(POSE_CLASSES)})"

    return JSON({
        "image": encode_data_url(frame),
        "status": status,
        "saved": list(_reg_gestures.keys()),
    })


output.register_callback("register_frame_cb", _register_frame_cb)

buttons_html = "".join(
    f'<button class="cls-btn" data-cls="{c}" onclick="selectClass(this.dataset.cls)" '
    f'style="margin:3px; padding:6px 10px; cursor:pointer;">{c}</button>'
    for c in POSE_CLASSES
)

display(HTML(f"""
<div style="font-family:sans-serif; max-width:720px;">
  <div style="margin-bottom:8px;">
    <strong>Class:</strong> <span id="sel-label" style="color:#0a7;">{POSE_CLASSES[0]}</span>
    &nbsp;|&nbsp;
    <label>Tolerance: <input id="tol" type="number" step="0.01" min="0.01" max="1.0" value="{TOLERANCE}" style="width:65px;"></label>
    &nbsp;|&nbsp;
    <button onclick="requestSave()" style="padding:6px 14px; background:#1976d2; color:#fff; border:none; cursor:pointer; font-weight:bold;">SAVE POSE</button>
    <button onclick="stopRegCam()" style="padding:6px 12px; background:#c62828; color:#fff; border:none; cursor:pointer;">STOP</button>
  </div>
  <div id="cls-bar" style="margin-bottom:8px;">{buttons_html}</div>
  <div id="reg-status" style="margin-bottom:6px; font-weight:bold;">Starting camera...</div>
  <video id="reg-vid" autoplay playsinline style="display:none;"></video>
  <img id="reg-img" style="width:100%; max-width:640px; border:2px solid #444;" />
</div>
<script>
(() => {{
  if (window._robodogStopCam) window._robodogStopCam();
  [window.activeDatasetStream, window.offlineStream, window.teleopStream].forEach(s => {{
    if (s) {{ try {{ s.getTracks().forEach(t => t.stop()); }} catch(e) {{}} }}
  }});
  window.activeDatasetStream = null;
  window.offlineStream = null;
  window.teleopStream = null;

  let selectedClass = "{POSE_CLASSES[0]}";
  let pendingSave = false;
  let running = true;
  let stream = null;
  const video = document.getElementById("reg-vid");
  const img = document.getElementById("reg-img");
  const statusEl = document.getElementById("reg-status");
  const canvas = document.createElement("canvas");
  canvas.width = 640;
  canvas.height = 480;
  const ctx = canvas.getContext("2d");

  window.selectClass = (cls) => {{
    selectedClass = cls;
    document.getElementById("sel-label").textContent = cls;
    refreshButtons([]);
  }};
  window.requestSave = () => {{ pendingSave = true; }};
  const stopFn = () => {{
    running = false;
    if (stream) stream.getTracks().forEach(t => t.stop());
    if (window._robodogStopCam === stopFn) window._robodogStopCam = null;
    statusEl.textContent = "Camera stopped.";
  }};
  window.stopRegCam = stopFn;
  window._robodogStopCam = stopFn;

  let lastSaved = [];
  function refreshButtons(savedList) {{
    if (savedList.length) lastSaved = savedList;
    document.querySelectorAll("#cls-bar .cls-btn").forEach(btn => {{
      const c = btn.dataset.cls;
      const isSel = c === selectedClass;
      const isSaved = lastSaved.includes(c);
      btn.style.background = isSel ? "#0288d1" : (isSaved ? "#2e7d32" : "#424242");
      btn.style.color = "#fff";
      btn.style.border = isSel ? "2px solid #ffeb3b" : "1px solid #888";
      btn.textContent = isSaved ? c + " *" : c;
    }});
  }}
  refreshButtons([]);

  async function loop() {{
    try {{
      stream = await navigator.mediaDevices.getUserMedia({{video: {{width: 640, height: 480}}}});
      video.srcObject = stream;
      await video.play();
      while (running) {{
        if (video.readyState < 2 || video.videoWidth === 0) {{
          await new Promise(r => setTimeout(r, 30));
          continue;
        }}
        ctx.save();
        ctx.translate(canvas.width, 0);
        ctx.scale(-1, 1);
        ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
        ctx.restore();
        const dataUrl = canvas.toDataURL("image/jpeg", 0.8);
        const doSave = pendingSave;
        pendingSave = false;
        const tol = parseFloat(document.getElementById("tol").value) || 0.15;
        const res = await google.colab.kernel.invokeFunction(
          "register_frame_cb", [dataUrl, selectedClass, doSave, tol], {{}}
        );
        let out = res?.data?.["application/json"];
        if (typeof out === "string") out = JSON.parse(out);
        if (out) {{
          if (out.image) img.src = out.image;
          statusEl.textContent = out.status;
          refreshButtons(out.saved || []);
        }}
        await new Promise(r => setTimeout(r, 30));
      }}
    }} catch (err) {{
      statusEl.textContent = "Camera error: " + err;
    }}
  }}
  loop();
}})();
</script>
"""))

### 4.1.3 Export: Download `gestures.json` (Optional)

Your `gestures.json` file is already saved in this Colab session and ready for **Section 4.2** below. Click **Play (`▶`)** if you also want to download a backup copy of `gestures.json` into your computer's **`Downloads`** folder.

In [ ]:
#@title 4.1.3 Download gestures.json (Optional) { display-mode: "form" }
# Optional: download gestures.json to your computer
from google.colab import files
if GESTURES_PATH.exists():
    files.download(str(GESTURES_PATH))
else:
    print("No gestures.json saved yet.")

---

## 4.2 Part 2: Lite3 Hand Pose Detection & REST Publisher

Now that you have recorded your hand shapes into `gestures.json` in Section 4.1, this section watches your webcam live, classifies which pose **class** you are making, and sends HTTP commands over the network to move the Lite3 robot dog.

### Key Concepts

- **What is a "Class"?**
  A named category for a hand pose. You saved up to 9 classes in `gestures.json`:
  - **7 movement classes** (`forward`, `backward`, `left`, `right`, `turn_left`, `turn_right`, `stop`) send speed commands 10 times per second while held.
  - **2 posture classes** (`sit`, `stand`) send a single posture trigger when you switch into that hand pose.
- **What is an API?** *(Application Programming Interface)*
  Think of the robot's API like a **receptionist or waiter**: instead of you needing to program the robot's leg motors directly, the robot runs a small server listening for polite command messages and forwards them to the robot's control system.
- **What is an HTTP Request (`POST`) & JSON Payload?**
  Just like your browser uses `http://` to fetch web pages, programs send commands across a network using **HTTP requests**. An **HTTP `POST` request** delivers an order slip formatted in **JSON** (text inside curly braces `{...}`):
  ```json
  {"topic": "/cmd_vel", "type": "geometry_msgs/Twist", "payload": {"linear": {"x": 0.2, "y": 0, "z": 0}, ...}}
  ```
  - `topic`: Which channel on the robot to send the command to (`/cmd_vel` for movement speeds, `/simple_cmd` for `sit`/`stand`).
  - `type`: The template name the robot expects (`geometry_msgs/Twist` or `message_transformer/SimpleCMD`).
  - `payload`: The actual numbers (speeds or command code).

### Beginner Step-by-Step Setup Checklist

1. **Confirm `gestures.json` is in Colab**:
   - If you completed **Section 4.1** in this same session, `gestures.json` is already in Colab's working directory.
   - If you recorded `gestures.json` earlier in a separate session, click the **folder icon (`Files`)** on the far-left sidebar of Colab, click **Upload to session storage**, and upload `gestures.json`.
2. **Join the Robot Wi-Fi & Check the Robot's Address**:
   - Connect your laptop to the workshop Wi-Fi and ask the instructor for the robot's **IP address** (e.g., `192.168.1.100`), or use `localhost` if the robot bridge runs on your laptop.
3. **Set `API_URL` in `4.2.1` Below**:
   - In `4.2.1`, update **only the address between `http://` and `:3000`**—keep `http://`, `:3000`, and `/api/topics/publish` intact:
   ```python
   BROWSER_LOCAL_FETCH = True
   API_URL = "https://API_ADDRESS.com/api/topics/publish"     # API hosted over the internet
   # or:
   API_URL = "http://192.168.1.100:3000/api/topics/publish"  # robot on local Wi-Fi (replace with your robot's IP)
   # or:
   API_URL = "http://localhost:3000/api/topics/publish"      # bridge running on your own laptop
   ```
4. **Handling Chrome 142+ Local Network Access (LNA) & Mixed Content** *(for LAN IPs like `172.16–31.x.x`, `192.168.x.x`, `10.x.x.x`, or `localhost`)*:
   - Google Colab runs cell outputs inside a cross-origin `*-colab.googleusercontent.com` `<iframe>` without `allow="local-network-access"`, so modern Chrome can block direct browser `fetch()` calls to private LAN IPs with `Permission was denied for this request to access the local address space`. Choose one of these two fixes:
   - **Option A (Recommended for Workshops — Zero Browser Flags):** Expose the robot bridge via an HTTPS tunnel (e.g., `cloudflared tunnel --url http://172.19.0.131:3000` or `ngrok http 3000`) and set `API_URL = "https://<your-tunnel-domain>/api/topics/publish"`. Set `BROWSER_LOCAL_FETCH = False` to send directly from Colab's Python kernel with zero browser CORS/LNA restrictions (or keep `True` if the bridge sends CORS headers).
   - **Option B (Direct LAN IP `http://192.168.x.x` / `http://172.16–31.x.x` / `http://10.x.x.x` in Chrome 142+):**
     1. Open `chrome://flags/#local-network-access-check`, set **Local Network Access checks** to **Disabled**, and click **Relaunch**.
     2. Click the **tune / lock icon** next to `colab.research.google.com` → **Site settings** → set **Insecure content** to **Allow** (and **Local network access** to **Allow** if listed), then reload Colab.

### Understanding the Robot Settings in `4.2.1`
- **`TOLERANCE` (`0.15`)**: How closely your live hand must match a saved pose. Lower (`0.10`) = stricter; higher (`0.20`) = easier to trigger.
- **`CLASS_PAYLOADS` (`/cmd_vel` Speeds)**:
  | Parameter | Controls | Positive (`+`) | Negative (`-`) |
  |---|---|---|---|
  | `lx` | Forward / backward speed | Walk `forward` (`0.2`) | Walk `backward` (`-0.2`) |
  | `ly` | Sideways step speed | Step `left` (`0.1`) | Step `right` (`-0.1`) |
  | `az` | Turning rotation speed | Turn `left` (`0.3`) | Turn `right` (`-0.3`) |
- **`ZERO_TWIST` Safety Brake**: When your hand leaves the frame or shows `Unknown`, the notebook sends `0.0` for all speeds so the robot stops safely instead of continuing its last motion.
- **`SIMPLE_CMD_PAYLOADS`**: `stand` sends command code `0x21010202` and `sit` sends `0x21010203` to `/simple_cmd` once each time you switch into that pose.

### 4.2.1 Configure `API_URL`, Topic Payloads & Tolerance

Edit **`API_URL`** on line 2 of the code cell below to point to your robot (replace `localhost` with the robot's IP address on Wi-Fi, e.g. `"http://192.168.1.100:3000/api/topics/publish"`), then click **Play (`▶`)**.

In [ ]:
#@title 4.2.1 Configure API_URL, Topic Payloads & Tolerance
# === API Endpoint & Twist Payload Configuration ===
API_URL = "http://localhost:3000/api/topics/publish"  # <-- Change 'localhost' to the robot's Wi-Fi IP if needed (e.g. 192.168.1.100)
BROWSER_LOCAL_FETCH = True  # True: browser sends HTTP POST over your local Wi-Fi (recommended in Colab); False: Python requests.post
TOPIC = "/cmd_vel"          # Robot channel for continuous walking/turning velocities
MSG_TYPE = "geometry_msgs/Twist"
SIMPLE_CMD_TOPIC = "/simple_cmd"  # Robot channel for one-shot posture commands (sit/stand)
SIMPLE_CMD_TYPE = "message_transformer/SimpleCMD"
STAND_CMD_CODE = 0x21010202  # 553714178 (stand up)
SIT_CMD_CODE = 0x21010203    # 553714179 (sit / lie down)

PUBLISH_HZ = 10
PUBLISH_INTERVAL = 1.0 / PUBLISH_HZ  # 0.1s between velocity commands (10 times per second)
HTTP_TIMEOUT = 0.1

from pathlib import Path
GESTURES_PATH = Path("gestures.json")  # Saved by Section 4.1 or uploaded to Colab Files panel
TOLERANCE = 0.15                       # Max difference score allowed to match a pose (lower = stricter)


def make_twist(lx: float = 0.0, ly: float = 0.0, lz: float = 0.0, az: float = 0.0) -> dict:
    return {
        "linear": {"x": lx, "y": ly, "z": lz},
        "angular": {"x": 0.0, "y": 0.0, "z": az},
    }


def make_simple_cmd(cmd_code: int, cmd_value: int = 0, cmd_type: int = 0) -> dict:
    return {"cmd_code": cmd_code, "cmd_value": cmd_value, "type": cmd_type}


CLASS_PAYLOADS: dict[str, dict] = {
    "forward": make_twist(lx=0.2),
    "backward": make_twist(lx=-0.2),
    "left": make_twist(ly=0.1),
    "right": make_twist(ly=-0.1),
    "turn_left": make_twist(az=0.3),
    "turn_right": make_twist(az=-0.3),
    "stop": make_twist(),
}
SIMPLE_CMD_PAYLOADS: dict[str, dict] = {
    "sit": make_simple_cmd(SIT_CMD_CODE),
    "stand": make_simple_cmd(STAND_CMD_CODE),
}
ZERO_TWIST = make_twist()
print(f"API configured: {API_URL} (BROWSER_LOCAL_FETCH={BROWSER_LOCAL_FETCH})")

### 4.2.2 Landmark Normalization, Gesture Classifier & HTTP Publisher

Click **Play (`▶`)** to load the HTTP request builder (and ensure the landmark helpers from `4.1.1` are ready), and check the output line printed below the cell:
- **`Loaded gestures: ['forward', ...]`**: Your `gestures.json` file was found and loaded!
- **`Loaded gestures: []`**: `gestures.json` is missing—record poses in **Section 4.1** or upload `gestures.json` using the 📁 **Files** sidebar before running `4.2.3`.

In [ ]:
#@title 4.2.2 Landmark Normalization, Gesture Classifier & HTTP Publisher { display-mode: "form" }
import base64
import json
import time
import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import requests
from mediapipe.tasks.python import vision

try:
    BaseOptions = mp.tasks.BaseOptions
except AttributeError:
    from mediapipe.tasks.python import core
    BaseOptions = core.base_options.BaseOptions

if "MODEL_PATH" not in globals():
    MODEL_PATH = Path("hand_landmarker.task")
    MODEL_URL = (
        "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
        "hand_landmarker/float16/1/hand_landmarker.task"
    )
    if not MODEL_PATH.exists():
        print("Downloading hand_landmarker.task...")
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

if "HAND_CONNECTIONS" not in globals():
    HAND_CONNECTIONS = (
        (0, 1), (1, 2), (2, 3), (3, 4),
        (0, 5), (5, 6), (6, 7), (7, 8),
        (5, 9), (9, 10), (10, 11), (11, 12),
        (9, 13), (13, 14), (14, 15), (15, 16),
        (13, 17), (17, 18), (18, 19), (19, 20),
        (0, 17),
    )

if "draw_hand_skeleton" not in globals():
    def draw_hand_skeleton(frame: np.ndarray, hand_landmarks) -> None:
        if hasattr(vision, "drawing_utils") and hasattr(vision, "drawing_styles") and hasattr(vision, "HandLandmarksConnections"):
            vision.drawing_utils.draw_landmarks(
                frame,
                hand_landmarks,
                vision.HandLandmarksConnections.HAND_CONNECTIONS,
                vision.drawing_styles.get_default_hand_landmarks_style(),
                vision.drawing_styles.get_default_hand_connections_style(),
            )
            return
        h, w = frame.shape[:2]
        pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]
        for a, b in HAND_CONNECTIONS:
            cv2.line(frame, pts[a], pts[b], (0, 220, 0), 2, cv2.LINE_AA)
        for x, y in pts:
            cv2.circle(frame, (x, y), 4, (0, 80, 255), -1, cv2.LINE_AA)

if "normalize_landmarks" not in globals():
    def normalize_landmarks(hand_landmarks) -> np.ndarray:
        coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks], dtype=np.float32)
        centered = coords - coords[0]
        scale = float(np.max(np.linalg.norm(centered, axis=1)))
        return centered / scale if scale > 1e-6 else centered

if "load_gestures" not in globals():
    def load_gestures(path: str | Path = GESTURES_PATH) -> dict[str, np.ndarray]:
        p = Path(path)
        if not p.exists():
            return {}
        raw = json.loads(p.read_text(encoding="utf-8"))
        return {name: np.asarray(coords, dtype=np.float32) for name, coords in raw.items()}

if "classify_gesture" not in globals():
    def classify_gesture(
        live_coords: np.ndarray,
        gestures: dict[str, np.ndarray],
        tolerance: float = TOLERANCE,
    ) -> tuple[str, float | None, str | None]:
        if not gestures:
            return "No gestures saved", None, None
        best_name, best_dist = None, float("inf")
        for name, ref_coords in gestures.items():
            if ref_coords.shape != live_coords.shape:
                continue
            dist = float(np.mean(np.linalg.norm(live_coords - ref_coords, axis=1)))
            if dist < best_dist:
                best_dist, best_name = dist, name
        if best_name is None:
            return "No gestures saved", None, None
        if best_dist <= tolerance:
            return best_name, best_dist, best_name
        return "Unknown", best_dist, best_name

if "decode_data_url" not in globals():
    def decode_data_url(data_url: str) -> np.ndarray | None:
        if not data_url or "," not in data_url:
            return None
        encoded = data_url.split(",", 1)[1]
        if not encoded:
            return None
        buf = np.frombuffer(base64.b64decode(encoded), dtype=np.uint8)
        if buf.size == 0:
            return None
        return cv2.imdecode(buf, cv2.IMREAD_COLOR)

if "encode_data_url" not in globals():
    def encode_data_url(frame: np.ndarray) -> str:
        ok, buf = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 80])
        if not ok:
            raise RuntimeError("Failed to encode frame")
        return "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode("ascii")

if "create_image_landmarker" not in globals():
    def create_image_landmarker() -> vision.HandLandmarker:
        global _shared_hand_landmarker
        if "_shared_hand_landmarker" in globals() and _shared_hand_landmarker is not None:
            return _shared_hand_landmarker
        if (
            "hands_detector" in globals()
            and getattr(hands_detector, "mode", None) == "tasks"
            and getattr(hands_detector, "detector", None) is not None
        ):
            _shared_hand_landmarker = hands_detector.detector
            return _shared_hand_landmarker
        options = vision.HandLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=str(MODEL_PATH)),
            running_mode=vision.RunningMode.IMAGE,
            num_hands=1,
        )
        _shared_hand_landmarker = vision.HandLandmarker.create_from_options(options)
        return _shared_hand_landmarker


def build_publish_body(class_name: str | None) -> tuple[dict, str]:
    if class_name in SIMPLE_CMD_PAYLOADS:
        topic, msg_type = SIMPLE_CMD_TOPIC, SIMPLE_CMD_TYPE
        payload, label = SIMPLE_CMD_PAYLOADS[class_name], class_name
    else:
        topic, msg_type = TOPIC, MSG_TYPE
        is_known = class_name in CLASS_PAYLOADS
        payload = CLASS_PAYLOADS[class_name] if is_known else ZERO_TWIST
        label = class_name if is_known else "zero"
    return {"topic": topic, "type": msg_type, "payload": payload}, label


def publish_cmd_vel(class_name: str | None) -> str:
    body, label = build_publish_body(class_name)
    try:
        resp = requests.post(
            API_URL,
            json=body,
            headers={
                "Accept": "application/json",
                "User-Agent": "robodog-workshop/1.0",
            },
            timeout=HTTP_TIMEOUT,
        )
        return f"HTTP {resp.status_code}: {label}"
    except requests.RequestException as exc:
        return f"HTTP err ({label}): {exc}"


print("Loaded gestures:", list(load_gestures().keys()))

### 4.2.3 Live Pose Detection & REST API Publish

Click **Play (`▶`)** on the cell below and allow webcam access to start controlling the robot dog live!

#### What Happens While Running
1. **Webcam & Hand Tracking**: Your browser captures mirrored `640x480` frames and sends them to MediaPipe, which locates 21 hand dots and draws your skeleton.
2. **Pose Matching**: Your live hand pose is compared against the classes you saved in `gestures.json`.
3. **Sending HTTP `POST` Commands (`BROWSER_LOCAL_FETCH = True`)**:
   - Python builds the JSON command payload and passes it back to **your browser**, which sends the HTTP `POST` request to `API_URL` directly over your local Wi-Fi or `localhost`.
   - **Movement classes** (`forward`, `backward`, `left`, `right`, `turn_left`, `turn_right`, `stop`) publish to `/cmd_vel` **10 times per second** (and send safe all-zero `ZERO_TWIST` speeds whenever no hand is seen or the pose is `Unknown`).
   - **Posture classes** (`sit`, `stand`) publish **once** to `/simple_cmd` each time your hand switches into `sit` or `stand`.

#### How to Read the Live Status Line
Look above or at the bottom of the video feed—for example: **`Gesture: forward (err=0.062) | Browser POST -> forward (HTTP 200)`**
- **`Gesture: forward (err=0.062)`**: Matched the `forward` class with distance error `0.062` (under your `TOLERANCE` of `0.15`).
- **`Browser POST -> forward (HTTP 200)`**: Your browser delivered the CORS-compliant HTTP `POST` request over the network and received a `2xx` confirmation from the robot bridge!
- Click **STOP CAMERA** when finished to turn off your webcam and return the robot to idle.

#### Beginner Troubleshooting Guide
- **`Loaded gestures: []` or `Gesture: No gestures saved`**: Colab cannot find `gestures.json` in the current session. Record poses in **Section 4.1**, or open the 📁 **Files** sidebar on the left, upload `gestures.json`, and re-run `4.2.2` + `4.2.3`.
- **`Gesture: Unknown`**: Your current hand shape didn't closely match any pose in `gestures.json` within `TOLERANCE`. Hold your pose steadier inside the frame, try a slightly higher `TOLERANCE` (like `0.20` in `4.2.1`), or re-record that class cleanly in **Section 4.1**.
- **`Browser CORS/LNA err ...` or `Permission was denied for this request to access the local address space`**: Your browser blocked the local-network request from Colab's output `<iframe>`, could not reach `API_URL`, or failed the CORS preflight (`OPTIONS`) check. Check that:
  1. **Easiest fix (Option A):** Run an HTTPS tunnel (e.g., `cloudflared tunnel --url http://172.19.0.131:3000`) and set `API_URL = "https://<your-tunnel-domain>/api/topics/publish"` (with `BROWSER_LOCAL_FETCH = False` to send from Python without browser CORS/LNA, or `True` if CORS headers are enabled).
  2. **Direct LAN IP fix (Option B):** Open `chrome://flags/#local-network-access-check` → set to **Disabled** → **Relaunch** Chrome; also allow **Insecure content** (and **Local network access** if listed) in **Site settings** for `colab.research.google.com`.
  3. The robot dog (and its REST API bridge on port `3000`) is powered on, your laptop is on the **same Wi-Fi**, and `API_URL` in `4.2.1` was re-run after editing.
  4. The server at `API_URL` handles `OPTIONS` preflight and `POST` requests with `Access-Control-Allow-Origin` (e.g., `*`), `Access-Control-Allow-Methods: POST, OPTIONS`, `Access-Control-Allow-Headers: Content-Type`, and `Access-Control-Allow-Private-Network: true`.
- **Camera not starting / already in use**: Click **STOP** on `4.1.2` (or close other video apps), allow webcam permission in the browser URL bar, and re-run `4.2.3`.

In [ ]:
#@title 4.2.3 Live Hand Pose Detection & REST API Publisher { display-mode: "form" }
from google.colab import output
from IPython.display import HTML, JSON, display

_det_landmarker = create_image_landmarker()
_det_gestures = load_gestures(GESTURES_PATH)
_last_pub_class = "__init__"
_last_pub_time = 0.0
_last_api_status = "API: idle"


def _detect_frame_cb(data_url: str, client_api_status: str = ""):
    global _det_gestures, _last_pub_class, _last_pub_time, _last_api_status
    _det_gestures = load_gestures(GESTURES_PATH)
    if BROWSER_LOCAL_FETCH and client_api_status:
        _last_api_status = client_api_status

    frame = decode_data_url(data_url)
    if frame is None:
        return JSON({"image": "", "status": f"Waiting for valid frame... | {_last_api_status}", "publish": None})

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = _det_landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))

    detected_label = None
    banner = "Hand: none"
    color = (180, 180, 180)

    if res.hand_landmarks:
        hand_lms = res.hand_landmarks[0]
        draw_hand_skeleton(frame, hand_lms)
        coords = normalize_landmarks(hand_lms)
        label, dist, *_ = classify_gesture(coords, _det_gestures, TOLERANCE)
        detected_label = label if (label in CLASS_PAYLOADS or label in SIMPLE_CMD_PAYLOADS) else None
        banner = f"Gesture: {label}" + (f" (err={dist:.3f})" if dist is not None else "")
        color = (0, 220, 0) if detected_label is not None else (0, 180, 255)

    now = time.monotonic()
    is_simple = detected_label in SIMPLE_CMD_PAYLOADS
    should_pub = (
        (detected_label != _last_pub_class)
        if is_simple
        else (detected_label != _last_pub_class or now - _last_pub_time >= PUBLISH_INTERVAL)
    )
    publish_req = None
    if should_pub:
        if BROWSER_LOCAL_FETCH:
            body, label = build_publish_body(detected_label)
            publish_req = {"url": API_URL, "body": body, "label": label}
            if not client_api_status:
                _last_api_status = f"Browser POST -> {label}"
        else:
            _last_api_status = publish_cmd_vel(detected_label)
        _last_pub_class = detected_label
        _last_pub_time = now

    cv2.putText(frame, banner, (16, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)
    cv2.putText(
        frame,
        _last_api_status[:55],
        (16, frame.shape[0] - 14),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.52,
        (220, 220, 220),
        1,
        cv2.LINE_AA,
    )
    return JSON({
        "image": encode_data_url(frame),
        "status": f"{banner} | {_last_api_status}",
        "publish": publish_req,
    })


output.register_callback("detect_frame_cb", _detect_frame_cb)

display(HTML("""
<div style="font-family:sans-serif; max-width:720px;">
  <div style="margin-bottom:8px;">
    <button onclick="stopDetCam()" style="padding:6px 14px; background:#c62828; color:#fff; border:none; cursor:pointer; font-weight:bold;">STOP CAMERA</button>
    <span id="det-status" style="margin-left:10px; font-weight:bold;">Starting camera...</span>
  </div>
  <video id="det-vid" autoplay playsinline style="display:none;"></video>
  <img id="det-img" style="width:100%; max-width:640px; border:2px solid #444;" />
</div>
<script>
(() => {
  if (window._robodogStopCam) window._robodogStopCam();
  [window.activeDatasetStream, window.offlineStream, window.teleopStream].forEach(s => {
    if (s) { try { s.getTracks().forEach(t => t.stop()); } catch(e) {} }
  });
  window.activeDatasetStream = null;
  window.offlineStream = null;
  window.teleopStream = null;

  let running = true;
  let stream = null;
  let lastApiStatus = "";
  const video = document.getElementById("det-vid");
  const img = document.getElementById("det-img");
  const statusEl = document.getElementById("det-status");
  const canvas = document.createElement("canvas");
  canvas.width = 640;
  canvas.height = 480;
  const ctx = canvas.getContext("2d");

  const stopFn = () => {
    running = false;
    if (stream) stream.getTracks().forEach(t => t.stop());
    if (window._robodogStopCam === stopFn) window._robodogStopCam = null;
    statusEl.textContent = "Camera stopped.";
  };
  window.stopDetCam = stopFn;
  window._robodogStopCam = stopFn;

  function inferTargetAddressSpace(rawUrl) {
    try {
      const host = new URL(rawUrl).hostname.toLowerCase();
      if (host === "localhost" || host === "[::1]" || host === "::1" || /^127\\./.test(host)) {
        return "loopback";
      }
      if (
        host.endsWith(".local") ||
        /^10\\./.test(host) ||
        /^192\\.168\\./.test(host) ||
        /^169\\.254\\./.test(host) ||
        /^172\\.(1[6-9]|2\\d|3[0-1])\\./.test(host)
      ) {
        return "local";
      }
    } catch (e) {}
    return null;
  }

  async function sendLocalApi(pub) {
    const payload = JSON.stringify(pub.body);
    const addrSpace = inferTargetAddressSpace(pub.url);
    const fetchOpts = {
      method: "POST",
      mode: "cors",
      credentials: "omit",
      headers: {
        "Content-Type": "application/json",
        "Accept": "application/json",
      },
      body: payload,
    };
    if (addrSpace) {
      fetchOpts.targetAddressSpace = addrSpace;
    }
    try {
      const resp = await fetch(pub.url, fetchOpts);
      lastApiStatus = resp.ok
        ? "Browser POST -> " + pub.label + " (HTTP " + resp.status + ")"
        : "Browser HTTP " + resp.status + ": " + pub.label;
    } catch (err) {
      lastApiStatus = "Browser CORS/LNA err (" + pub.label + "): " + (err?.message || err);
    }
  }

  async function loop() {
    try {
      stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
      video.srcObject = stream;
      await video.play();
      while (running) {
        if (video.readyState < 2 || video.videoWidth === 0) {
          await new Promise(r => setTimeout(r, 30));
          continue;
        }
        ctx.save();
        ctx.translate(canvas.width, 0);
        ctx.scale(-1, 1);
        ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
        ctx.restore();
        const dataUrl = canvas.toDataURL("image/jpeg", 0.8);
        const res = await google.colab.kernel.invokeFunction("detect_frame_cb", [dataUrl, lastApiStatus], {});
        let out = res?.data?.["application/json"];
        if (typeof out === "string") out = JSON.parse(out);
        if (out) {
          if (out.image) img.src = out.image;
          statusEl.textContent = out.status;
          if (out.publish) sendLocalApi(out.publish);
        }
        await new Promise(r => setTimeout(r, 30));
      }
    } catch (err) {
      statusEl.textContent = "Camera error: " + err;
    }
  }
  loop();
})();
</script>
"""))

---

## 4.3 Part 3: Natural-Language LLM Control for Lite3 RoboDog

> **Quick Goal:** Replace manual hand poses with everyday English or Indonesian sentences (such as `"Move forward"`, `"Turn left"`, or `"Sit down"`), using prompt engineering to translate intent into structured `geometry_msgs/Twist` payloads while safely rejecting impossible requests.

In **Section 4.2**, you controlled the Lite3 RoboDog with discrete hand poses matched against `gestures.json`. Earlier in **Chapter 2.4** you learned how a **system prompt** shapes LLM output, and in **Section 3.8** you sent natural-language tasks to the UR5 where the rules were locked on the lab server. Here, **you engineer the robot's rulebook yourself**!

| Aspect | 4.2 Hand Pose Teleop | 3.8 UR5 LLM Control | 4.3 RoboDog LLM Control |
| --- | --- | --- | --- |
| **Input Modality** | Webcam hand landmarks | Natural-language task | Natural-language command |
| **System Prompt** | N/A (nearest-neighbor match) | Read-only on lab server | Student-engineered (`SYSTEM_PROMPT` + `FEW_SHOT_EXAMPLES`) |
| **Output Format** | Discrete class label | Server-side action step | Friendly reply + `geometry_msgs/Twist` JSON |
| **Transport** | Browser `fetch` → `API_URL` | `RobotSession` WebSocket | Cloud `LLM_URL` → Browser `fetch` → `API_URL` |

### Key Educational Concepts

* **System Prompt + Few-Shot Prompting + `TEMPERATURE = 0.0`:**
  * **`SYSTEM_PROMPT`**: Defines the robot's capability boundary, exact `linear`/`angular` velocity conventions, and when to refuse.
  * **`FEW_SHOT_EXAMPLES`**: Concrete user/assistant demonstration pairs placed before your prompt so the model reliably reproduces the two-part output format (one conversational sentence + one raw JSON object).
  * **`TEMPERATURE = 0.0`**: Sets sampling randomness to zero so identical commands yield deterministic, predictable motion parameters—essential for physical hardware safety.
* **Safety Refusal Guardrails (Parser Gate):**
  * Unsafe or impossible commands (e.g., `"Fly to the ceiling"`, `"Jump over the table"`, `"Chase that person"`) must return a polite refusal with **no JSON at all**.
  * Our parser (`split_reply_and_twist()`) only emits a robot payload when a valid `{"linear": ..., "angular": ...}` JSON object is present—if no JSON is found, **zero packets** are sent to `/cmd_vel`.
* **Two-Stage Hybrid Network Flow:**
  * Stage 1 (Cloud → Cloud): Colab's Python kernel calls the OpenAI-compatible LLM endpoint (`LLM_URL`) with your `Authorization: Bearer` workshop passkey (`LLM_API_KEY`).
  * Stage 2 (Browser → Local Wi-Fi): With `BROWSER_LOCAL_FETCH = True`, Colab returns the parsed Twist JSON to your browser, which sends the HTTP `POST` over local Wi-Fi to the same `API_URL` bridge used in Section 4.2 (using Chrome's Local Network Access / *Insecure content* settings for LAN IPs, or an HTTPS tunnel with `BROWSER_LOCAL_FETCH = False`).

### Robot Velocity Table (`geometry_msgs/Twist`)

| Field | Controls | Positive (`+`) | Negative (`-`) |
|---|---|---|---|
| `linear.x` | Forward / backward speed | Walk `forward` (`1.0`) | Walk `backward` (`-1.0`) |
| `linear.y` | Sideways step speed | Step `left` (`1.0`) | Step `right` (`-1.0`) |
| `angular.z` | Turning rotation speed | Turn `left` (`0.5`) | Turn `right` (`-0.5`) |
| `linear.z` | Posture height target | `stand` (`0.1`) | `sit` (`-0.1`) |
| *(all `0.0`)* | Safety stop | `stop` / halt / freeze | — |

### 4.3.1 Configure LLM Endpoint & Robot API Settings

Run this cell to configure the LLM endpoint (`LLM_URL`, `MODEL_ID`, `TEMPERATURE = 0.0`) and robot bridge topic (`CMD_VEL_TOPIC = "/cmd_vel"`).

* **Automatic Reuse from Section 4.2:** If you already updated `API_URL` (e.g., `http://192.168.1.100:3000/api/topics/publish`) or `BROWSER_LOCAL_FETCH` in **Cell 4.2.1**, this cell automatically keeps those values so you do not have to re-type your robot's Wi-Fi IP.
* If you jumped straight to Section 4.3, edit `API_URL` directly below before running.

In [ ]:
#@title 4.3.1 Configure LLM Endpoint & Robot API Settings
# =============================================================================
# 1. EASY CONFIGURATION (Edit these values if needed)
# =============================================================================

LLM_URL = "https://llm.fawzqi.my.id/v1/chat/completions"  # Public OpenAI-compatible AI endpoint
LLM_API_KEY = "b4workshop"                                # Workshop passkey sent as Bearer token
MODEL_ID = "qwen3.5:9b"
REASONING_EFFORT = "none"                                 # Fast direct answers without hidden thinking tokens
TEMPERATURE = 0.0                                         # 0.0 = consistent, predictable robot commands

# Reuse API_URL / BROWSER_LOCAL_FETCH from 4.2.1 if already configured; otherwise default to localhost
API_URL = globals().get("API_URL", "http://localhost:3000/api/topics/publish")
BROWSER_LOCAL_FETCH = globals().get("BROWSER_LOCAL_FETCH", True)
CMD_VEL_TOPIC = "/cmd_vel"                                # Robot channel for velocity/motion commands
CMD_VEL_TYPE = "geometry_msgs/Twist"                      # ROS message structure expected by the robot bridge

print(f"Configuration ready. Model: {MODEL_ID} | Robot API_URL: {API_URL} (BROWSER_LOCAL_FETCH={BROWSER_LOCAL_FETCH})")

### 4.3.2 Engineer Your System Prompt & Few-Shot Examples

Just like in **Chapter 2.4**, the **System Prompt** acts as the rule layer that governs every turn, while **Few-Shot Examples** anchor the exact output structure:
- **`SYSTEM_PROMPT`**: Defines the valid actions, exact `linear`/`angular` velocity values, bilingual behavior (English or Indonesian), and the refusal rule for unsafe/impossible commands.
- **`FEW_SHOT_EXAMPLES`**: Pairs 9 valid movement/posture commands and 3 refusal examples so the LLM learns both *how to format JSON* and *when NOT to emit JSON*.

> **Try It:** Experiment with adjusting the speed numbers in `SYSTEM_PROMPT` (or adding an Indonesian few-shot example like `"Maju pelan"`), then click **Play (`▶`)** to load your prompt.

In [ ]:
#@title 4.3.2 System Prompt Engineering & Few-Shot Examples
# =============================================================================
# 2. SYSTEM PROMPT ENGINEERING & FEW-SHOT EXAMPLES
# =============================================================================

# TODO: Engineer your system prompt here.
# Customize movement speeds, coordinate conventions, and response style.
SYSTEM_PROMPT = """
You are a friendly assistant that controls a robot dog (robodog).
Translate the user's command into a ROS geometry_msgs/Twist JSON message.

Response rules:
1. If the command contains a VALID action (forward, backward, move/strafe left, move/strafe right, turn/rotate left, turn/rotate right, stand/stand up, sit/sit down, stop/halt/freeze):
   - On the first line, reply with a short, friendly confirmation in English (for example: "Understood, moving forward."). If the user writes in Indonesian, you may reply in Indonesian.
   - On the next line, output EXACTLY ONE valid JSON object with NO markdown code fences and NO extra text after the JSON:
     {"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}
   - Velocity conventions:
     * forward / move forward / walk forward: linear.x = 1.0
     * backward / move backward / walk back: linear.x = -1.0
     * move left / strafe left / step left: linear.y = 1.0
     * move right / strafe right / step right: linear.y = -1.0
     * turn left / rotate left / spin left: angular.z = 0.5
     * turn right / rotate right / spin right: angular.z = -0.5
     * stand / stand up: linear.z = 0.1
     * sit / sit down / lie down: linear.z = -0.1
     * stop / halt / freeze / stay still: all values 0.0
2. If the command is INVALID, unsafe, impossible for a robot dog, or unclear:
   - Do NOT output any JSON at all.
   - Reply politely with: "I cannot do that." (or in the user's language).
""".strip()

# TODO: Add your few-shot examples here.
# Each entry pairs a sample "user" message with the expected "model" reply.
FEW_SHOT_EXAMPLES: list[dict[str, str]] = [
    {
        "user": "Move forward",
        "model": 'Understood, moving forward.\n{"linear": {"x": 1.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Walk backward",
        "model": 'Understood, walking backward.\n{"linear": {"x": -1.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Strafe left",
        "model": 'Understood, moving to the left.\n{"linear": {"x": 0.0, "y": 1.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Move to the right",
        "model": 'Understood, moving to the right.\n{"linear": {"x": 0.0, "y": -1.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Turn left",
        "model": 'Understood, turning left.\n{"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.5}}',
    },
    {
        "user": "Turn right",
        "model": 'Understood, turning right.\n{"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": -0.5}}',
    },
    {
        "user": "Stand up",
        "model": 'Understood, standing up.\n{"linear": {"x": 0.0, "y": 0.0, "z": 0.1}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Sit down",
        "model": 'Understood, sitting down.\n{"linear": {"x": 0.0, "y": 0.0, "z": -0.1}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Stop",
        "model": 'Understood, stopping now.\n{"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}',
    },
    {
        "user": "Fly up into the sky",
        "model": "I cannot do that.",
    },
    {
        "user": "Jump over the table",
        "model": "I cannot do that.",
    },
    {
        "user": "Chase that person",
        "model": "I cannot do that.",
    },
]

print(f"Prompt ready with {len(FEW_SHOT_EXAMPLES)} few-shot examples.")

### 4.3.3 Load LLM Client, JSON Extractor & Robot Publisher Helpers

Click **Play (`▶`)** to load the five core helper functions powering the pipeline (`User Text` ➔ `LLM API` ➔ `JSON Parser Gate` ➔ `REST Bridge`):
- **`build_messages()`**: Packs `SYSTEM_PROMPT`, `FEW_SHOT_EXAMPLES`, and your command into an ordered chat list.
- **`ask_llm()`**: Sends that chat list to `LLM_URL` via an HTTP `POST` with your `Authorization: Bearer` key and returns the AI's raw text.
- **`split_reply_and_twist()`**: Separates the polite user-facing message from the JSON object (or returns `None` when the command was refused).
- **`build_robot_body()` / `publish_twist_payload()`**: Wraps the extracted Twist payload into `{"topic": "/cmd_vel", "type": "geometry_msgs/Twist", "payload": ...}` for your browser (or Python kernel) to send.

In [ ]:
#@title 4.3.3 Load LLM Client, JSON Extractor & Robot Publisher Helpers { display-mode: "form" }
import json
import re
import urllib.error
import urllib.request


def build_messages(
    user_text: str,
    system_prompt: str = SYSTEM_PROMPT,
    examples: list[dict[str, str]] = FEW_SHOT_EXAMPLES,
) -> list[dict[str, str]]:
    """Build chat messages with system prompt, few-shot examples, and user input."""
    messages = [{"role": "system", "content": system_prompt}]
    for example in examples:
        messages.append({"role": "user", "content": example["user"]})
        messages.append({"role": "assistant", "content": example["model"]})
    messages.append({"role": "user", "content": user_text})
    return messages


def ask_llm(
    user_text: str,
    model_id: str = MODEL_ID,
    llm_url: str = LLM_URL,
    api_key: str = LLM_API_KEY,
    reasoning_effort: str = REASONING_EFFORT,
    temperature: float = TEMPERATURE,
    timeout: float = 30.0,
) -> str:
    """Send a chat completion request to the LLM endpoint and return assistant text."""
    body = json.dumps({
        "model": model_id,
        "reasoning_effort": reasoning_effort,
        "temperature": temperature,
        "messages": build_messages(user_text),
    }).encode("utf-8")
    req = urllib.request.Request(
        llm_url,
        data=body,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "User-Agent": "robodog-workshop/1.0",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    return str(data["choices"][0]["message"]["content"]).strip()


def split_reply_and_twist(text: str) -> tuple[str, dict[str, dict[str, float]] | None]:
    """Separate user-facing reply text from an embedded Twist JSON payload."""
    decoder = json.JSONDecoder()
    idx = 0
    while (start := text.find("{", idx)) != -1:
        try:
            data, end_offset = decoder.raw_decode(text[start:])
        except json.JSONDecodeError:
            idx = start + 1
            continue

        if (
            isinstance(data, dict)
            and isinstance(data.get("linear"), dict)
            and isinstance(data.get("angular"), dict)
        ):
            twist = {
                section: {axis: float(data[section].get(axis, 0.0)) for axis in ("x", "y", "z")}
                for section in ("linear", "angular")
            }
            end = start + end_offset
            without_json = (text[:start] + text[end:]).strip()
            cleaned = re.sub(r"```(?:json)?\s*```", "", without_json).strip()
            return (cleaned or "Understood."), twist

        idx = start + 1

    return (text.strip() or "I cannot do that."), None


def build_robot_body(payload: dict[str, dict[str, float]]) -> dict:
    """Wrap a Twist payload into the REST bridge publish format."""
    return {
        "topic": CMD_VEL_TOPIC,
        "type": CMD_VEL_TYPE,
        "payload": payload,
    }


def publish_twist_payload(
    payload: dict[str, dict[str, float]],
    api_url: str = API_URL,
    timeout: float = 0.5,
) -> tuple[bool, str]:
    """POST a geometry_msgs/Twist JSON payload directly from the Python kernel."""
    body = json.dumps(build_robot_body(payload)).encode("utf-8")
    req = urllib.request.Request(
        api_url,
        data=body,
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json",
            "User-Agent": "robodog-workshop/1.0",
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return True, f"HTTP {resp.status}"
    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        return False, f"HTTP error: {exc}"


print("Helpers loaded and ready.")

### 4.3.4 Interactive Natural-Language Command Console

Click **Play (`▶`)** on the cell below to launch the interactive robot command console! (If the webcam loop from `4.2.3` was still running, this cell automatically stops it so hand-pose `ZERO_TWIST` frames do not override your LLM commands.)

#### How to Use the Console
1. Type a natural-language command into the box and press **Enter** (or click **Send**):
   - **Valid tests:** `Move forward`, `Turn left`, `Sit down`, `Stop` (or Indonesian commands such as `Jalan maju` or `Berhenti`).
   - **Safety refusal test:** `Fly to the ceiling` *(should politely refuse and print no `Payload:` or `Published to` lines)*.
2. Inspect the console log lines after each command:
   - **`You: ...`**: What you typed.
   - **`Robot: ...`**: The friendly reply shown to the user (with raw JSON cleanly stripped).
   - **`Payload: {"linear": ..., "angular": ...}`**: Exact Twist JSON payload sent to the robot (only printed when a valid command was parsed).
   - **`Published to <API_URL> -> HTTP 200`**: Confirmation that your browser (or Python kernel) delivered the HTTP `POST` request.
3. Click **Clear** anytime to wipe the console log.

> **Class Discussion Question:** *Notice that `split_reply_and_twist()` acts as a parser gate between the LLM and the robot motors. What happens if the LLM ignores the formatting rule and returns conversational text without valid `linear`/`angular` keys? Why is failing closed (sending nothing) safer than guessing?*

#### Troubleshooting
- **`LLM error: ...`**: Check your internet connection and verify `LLM_URL` and `LLM_API_KEY` in `4.3.1`, then re-run `4.3.1`–`4.3.4`.
- **`Published to ... -> CORS/LNA err ...` or `Permission was denied for this request to access the local address space`**:
  1. **Option A (Recommended):** Expose the bridge via an HTTPS tunnel (`cloudflared`/`ngrok`), set `API_URL` to `https://<your-tunnel-domain>/api/topics/publish`, and set `BROWSER_LOCAL_FETCH = False` in `4.3.1` (or keep `True` if CORS headers are configured).
  2. **Option B (Direct LAN IP in Chrome 142+):** Set `chrome://flags/#local-network-access-check` to **Disabled** and relaunch Chrome, allow **Insecure content** (and **Local network access** if listed) in **Site settings** for `colab.research.google.com`, confirm your laptop is on the same Wi-Fi as the robot, and ensure the server at `API_URL` responds to `OPTIONS` preflight and `POST` requests with `Access-Control-Allow-Origin`, `Access-Control-Allow-Methods: POST, OPTIONS`, `Access-Control-Allow-Headers: Content-Type`, and `Access-Control-Allow-Private-Network: true`.
- **AI misunderstands a command or picks the wrong speed**: Adjust the wording or add a matching example in `SYSTEM_PROMPT` / `FEW_SHOT_EXAMPLES` (`4.3.2`), then re-run `4.3.2`, `4.3.3`, and `4.3.4`.

In [ ]:
#@title 4.3.4 Interactive LLM Robot Command Console { display-mode: "form" }
from google.colab import output
from IPython.display import HTML, JSON, display


def _llm_chat_cb(user_text: str):
    text = (user_text or "").strip()
    if not text:
        return JSON({"error": "Please enter a command."})
    try:
        reply = ask_llm(text)
    except (urllib.error.URLError, TimeoutError, OSError, KeyError, IndexError, json.JSONDecodeError) as exc:
        return JSON({"error": f"LLM error: {exc}"})

    user_message, payload = split_reply_and_twist(reply)
    publish_req = None
    api_status = None
    if payload is not None:
        if BROWSER_LOCAL_FETCH:
            publish_req = {"url": API_URL, "body": build_robot_body(payload)}
        else:
            _, api_status = publish_twist_payload(payload, api_url=API_URL)

    return JSON({
        "user_message": user_message,
        "payload": payload,
        "publish": publish_req,
        "api_url": API_URL,
        "api_status": api_status,
    })


output.register_callback("llm_chat_cb", _llm_chat_cb)

display(HTML("""
<div style="font-family:sans-serif; max-width:760px;">
  <div id="llm-status" style="margin-bottom:8px; font-weight:bold;">Ready! Type a robot command below.</div>
  <div id="llm-log" style="background:#181818; color:#eee; font-family:monospace; font-size:13px; padding:12px; height:280px; overflow-y:auto; border:1px solid #444; border-radius:4px; white-space:pre-wrap;"></div>
  <div style="display:flex; gap:8px; margin-top:8px;">
    <input id="llm-input" type="text" placeholder="Type a robot command (e.g. Move forward, Turn left, Sit down)..." style="flex:1; padding:8px; font-size:14px;" />
    <button id="llm-send" style="padding:8px 16px; background:#1976d2; color:#fff; border:none; cursor:pointer; font-weight:bold;">Send</button>
    <button id="llm-clear" style="padding:8px 12px; background:#555; color:#fff; border:none; cursor:pointer;">Clear</button>
  </div>
</div>
<script>
(() => {
  if (window._robodogStopCam) window._robodogStopCam();

  const statusEl = document.getElementById("llm-status");
  const logEl = document.getElementById("llm-log");
  const inputEl = document.getElementById("llm-input");
  const sendBtn = document.getElementById("llm-send");
  const clearBtn = document.getElementById("llm-clear");

  function appendLine(text, color = "#eee") {
    const line = document.createElement("div");
    line.style.color = color;
    line.textContent = text;
    logEl.appendChild(line);
    logEl.scrollTop = logEl.scrollHeight;
  }

  function inferTargetAddressSpace(rawUrl) {
    try {
      const host = new URL(rawUrl).hostname.toLowerCase();
      if (host === "localhost" || host === "[::1]" || host === "::1" || /^127\\./.test(host)) {
        return "loopback";
      }
      if (
        host.endsWith(".local") ||
        /^10\\./.test(host) ||
        /^192\\.168\\./.test(host) ||
        /^169\\.254\\./.test(host) ||
        /^172\\.(1[6-9]|2\\d|3[0-1])\\./.test(host)
      ) {
        return "local";
      }
    } catch (e) {}
    return null;
  }

  async function sendLocalApi(pub) {
    const payload = JSON.stringify(pub.body);
    const addrSpace = inferTargetAddressSpace(pub.url);
    const fetchOpts = {
      method: "POST",
      mode: "cors",
      credentials: "omit",
      headers: {
        "Content-Type": "application/json",
        "Accept": "application/json",
      },
      body: payload,
    };
    if (addrSpace) {
      fetchOpts.targetAddressSpace = addrSpace;
    }
    try {
      const resp = await fetch(pub.url, fetchOpts);
      return resp.ok ? "HTTP " + resp.status : "HTTP " + resp.status + " (error)";
    } catch (err) {
      if (addrSpace) {
        return "CORS/LNA err: " + (err?.message || err) + " — Chrome blocked Colab iframe access to the local network; use an HTTPS tunnel or disable chrome://flags/#local-network-access-check";
      }
      return "CORS/HTTP err: " + (err?.message || err);
    }
  }

  async function submitCommand() {
    const cmd = inputEl.value.trim();
    if (!cmd) return;
    inputEl.value = "";
    sendBtn.disabled = true;
    statusEl.textContent = "Sending command to AI...";
    appendLine("\\nYou: " + cmd, "#90caf9");
    try {
      const res = await google.colab.kernel.invokeFunction("llm_chat_cb", [cmd], {});
      let out = res?.data?.["application/json"];
      if (typeof out === "string") out = JSON.parse(out);
      if (!out || out.error) {
        appendLine(out?.error || "Unknown callback error", "#ef9a9a");
      } else {
        appendLine("Robot: " + out.user_message, "#a5d6a7");
        if (out.payload) {
          appendLine("Payload: " + JSON.stringify(out.payload), "#ffe082");
          if (out.publish) {
            const apiStatus = await sendLocalApi(out.publish);
            appendLine("Published to " + out.publish.url + " -> " + apiStatus, "#ce93d8");
          } else if (out.api_status) {
            appendLine("Published to " + out.api_url + " -> " + out.api_status, "#ce93d8");
          }
        }
      }
    } catch (err) {
      appendLine("Error: " + err, "#ef9a9a");
    } finally {
      sendBtn.disabled = false;
      statusEl.textContent = "Ready! Type a robot command below.";
      inputEl.focus();
    }
  }

  sendBtn.addEventListener("click", submitCommand);
  inputEl.addEventListener("keydown", (e) => {
    if (e.key === "Enter") submitCommand();
  });
  clearBtn.addEventListener("click", () => { logEl.textContent = ""; });
})();
</script>
"""))

---

## 4.4 Workshop Summary & Final Checklist

## Human-Robot Interaction Workshop Final Checklist
- [x] Chapter 0: Environment dependencies and detection model setup.
- [x] Chapter 1: Pose-posture feature engineering and Machine Learning model.
- [x] Chapter 2: High-level UR5 commands using LLMs.
- [x] Chapter 3: Manipulator waypoint teleoperation and safety control.
- [x] Chapter 4: Lite3 RoboDog hand pose teleoperation (`gestures.json` + REST API) and natural-language LLM control.